In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import pyodbc
from sqlalchemy import create_engine, event
from urllib.parse import quote_plus

import logging

pd.set_option('display.max_columns', None)

In [44]:
FECHA_INICIO: str = "20210101"
CAMPANIA: int = 2021

BASE = Path( Path.home() / "Camposol S.A" / "Area Proyeccion Planif. O - Documentos" / "1. Arandano")

ruta_peso_real         = BASE / "2. SEGUIMIENTO_COSECHA" / "01. Cartilla de seguimiento diario" / "2026" / "Reporte_Pesos_Visualizacion" / "Calidad_Data_Peso_Variedad.xlsx"
# ruta_historial_siembra = BASE / "9. ANALISTA_PROYECCION" / "Historial de Siembra Arándano para Calidad V79.xlsx"
ruta_historico_poda    = BASE / "9. ANALISTA_PROYECCION" / "historico_de_poda.xlsx"
ruta_maestro_fechas    = BASE / "9. ANALISTA_PROYECCION" / "Maestro.xlsx"
ruta_resultado_final = BASE / "9. ANALISTA_PROYECCION" / "6. ENRIQUE_MOSQUEIRA" / "projects" / "df_resultado_final.xlsx"

for archivo in [
    ruta_peso_real,
    # ruta_historial_siembra,
    ruta_historico_poda,
    ruta_maestro_fechas,
]:
    if not archivo.exists():
        raise FileNotFoundError(f"No existe el archivo: {archivo}")

In [3]:
# Funciones auxiliares:

def validar_unicidad(df, claves, nombre_df="DataFrame"):
    dup = (
        df.groupby(claves, dropna=False)
        .size()
        .reset_index(name="n")
        .query("n > 1")
    )

    if not dup.empty:
        print(dup)
        raise ValueError(
            f"Hay duplicados en {nombre_df} para las claves: {', '.join(claves)}"
        )


def validar_cruce(df_orig, df_final, proceso="Cruce"):
    len_o, len_f = len(df_orig), len(df_final)
    diff = len_f - len_o

    if diff == 0:
        print(f"✅ [{proceso}] Validación exitosa: El número de filas se mantuvo intacto en {len_o}.")
        return # No retorna nada, evitando el print automático de Jupyter

    pct = diff / len_o
    tipo_error = "Duplicación" if diff > 0 else "Pérdida"
    emoji = "⚠️" if diff > 0 else "🚨"

    error_msg = (
        f"{emoji} [{proceso}] Error de {tipo_error} detectado. "
        f"Se esperaban {len_o} filas pero se obtuvieron {len_f} ({'+' if diff > 0 else ''}{diff} filas, {pct:.1%})."
    )

    raise ValueError(error_msg)

In [4]:
# Conexión a Base de datos local (CMPPRONOSTICOS)

server = 'CSPLCDB02\\PRDFRACTAL'
database = 'CMPPRONOSTICOS'
username = 'proyeccion'
password = 'Pr0yecci0n19'

# Lee drivers instalados y se queda con el último que sea de SQL Server
drivers = [d for d in pyodbc.drivers() if "SQL Server" in d]
if not drivers:
    raise RuntimeError(f"No se encontró ningún driver ODBC de SQL Server. Instalados: {pyodbc.drivers()}")

driver = drivers[-1]  # suele ser el más reciente

# String de conexión ODBC
conn_str = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={{{password}}};"
    f"Encrypt=yes;"
    f"TrustServerCertificate=yes;"
)

# Para SQLAlchemy, mejor codificar el string
engine_proy = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}")
print("Conexión a base de datos establecida exitosamente.")
print("Driver usado:", driver)

Conexión a base de datos establecida exitosamente.
Driver usado: ODBC Driver 17 for SQL Server


In [ ]:
# Conexión a Base de datos Cloud (COSTOS)

server = 'csolbibdportal.public.5f82c723813c.database.windows.net,3342'
database = 'CSOL_BD_COSTOS_CAMPO'
username = 'CON_PE_COS_BD_GALEGRIA'
password = '@ProyARD2026##'

# Lee drivers instalados y se queda con el último que sea de SQL Server
drivers = [d for d in pyodbc.drivers() if "SQL Server" in d]
if not drivers:
    raise RuntimeError(f"No se encontró ningún driver ODBC de SQL Server. Instalados: {pyodbc.drivers()}")

driver = drivers[-1]  # suele ser el más reciente

# String de conexión ODBC
conn_str = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={{{password}}};"
    f"Encrypt=yes;"
    f"TrustServerCertificate=yes;"
)

# Para SQLAlchemy, mejor codificar el string
engine_costos = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}")
print("Conexión a base de datos establecida exitosamente.")
print("Driver usado:", driver)

Conexión a base de datos establecida exitosamente.
Driver usado: ODBC Driver 17 for SQL Server


In [6]:
query_calendario_juliano = f"""
SELECT
    cj.Fecha,
    cj.AñoJuliano            AS Año,
    cj.SemanaJuliano         AS Semana,
    cj.AñoSemana,                              -- '202602', llave/etiqueta semanal
    cj.MesPorSemana          AS Mes,
    cj.NombreMesPorSemana    AS NombreMes,
    MIN(cj.Fecha) OVER (PARTITION BY cj.AñoJuliano, cj.SemanaJuliano) AS FechaInicioSemana,
    MAX(cj.Fecha) OVER (PARTITION BY cj.AñoJuliano, cj.SemanaJuliano) AS FechaFinSemana,
    COUNT(*)      OVER (PARTITION BY cj.AñoJuliano, cj.SemanaJuliano) AS Dias_En_Semana
FROM CalendarioJuliano cj
WHERE cj.Fecha >= '{CAMPANIA - 1}-01-01'
"""

with engine_proy.begin() as conn:
    df_calendario_juliano = (
        pd.read_sql(query_calendario_juliano, conn)
        .assign(
            Fecha = lambda d: pd.to_datetime(d['Fecha']),
            FechaInicioSemana=lambda d: pd.to_datetime(d["FechaInicioSemana"]),
            FechaFinSemana=lambda d: pd.to_datetime(d["FechaFinSemana"]),
            Año=lambda d: pd.to_numeric(d['Año'], errors='coerce'),
            Mes=lambda d: pd.to_numeric(d['Mes'], errors='coerce'),
            Semana=lambda d: pd.to_numeric(d['Semana'], errors='coerce'),
            AñoSemana=lambda d: pd.to_numeric(d['AñoSemana'], errors='coerce'),
        )
        .sort_values(['Fecha'])
        # Índice global de semana (mismo valor para todos los días de la semana) factorize calcula el índice correlativo directamente por fila
        .assign(
            Semana_Correlativa=lambda d: pd.factorize(d['AñoSemana'], sort=True)[0] + 1
        )
        .reset_index(drop=True)
    )

df_calendario_juliano.head()

,Fecha,Año,Semana,AñoSemana,Mes,NombreMes,FechaInicioSemana,FechaFinSemana,Dias_En_Semana,Semana_Correlativa
0,2020-01-01,2020,1,2001,1,Enero,2020-01-01,2020-01-05,5,1
1,2020-01-02,2020,1,2001,1,Enero,2020-01-01,2020-01-05,5,1
2,2020-01-03,2020,1,2001,1,Enero,2020-01-01,2020-01-05,5,1
3,2020-01-04,2020,1,2001,1,Enero,2020-01-01,2020-01-05,5,1
4,2020-01-05,2020,1,2001,1,Enero,2020-01-01,2020-01-05,5,1


In [7]:
df_maestro_fechas = (
    pd.read_excel(ruta_maestro_fechas, sheet_name = 'Parcela2')
    .dropna(subset=["Campaña", "SubParcela"])
    .assign(
        Campaña=lambda d: d["Campaña"].str.extract(r"C(\d{2})")[0].astype(int) + 2000,
        FechaIniCos_M=lambda d: pd.to_datetime(d["FechaIniCos"], dayfirst=True),
        FechaFinCos_M=lambda d: pd.to_datetime(d["FechaFinCos"], dayfirst=True),
    )
    [["Campaña", "SubParcela", "FechaIniCos_M", "FechaFinCos_M"]]
)

validar_unicidad(df_maestro_fechas, ["Campaña", "SubParcela"], "df_maestro_fechas")

df_maestro_fechas.head()

,Campaña,SubParcela,FechaIniCos_M,FechaFinCos_M
0,2021,P03,2021-08-20,2022-06-23
1,2022,P03,2022-12-21,2023-06-15
2,2023,P03,2023-11-16,2024-06-14
3,2024,P03,2024-12-09,2025-07-01
4,2025,P03,2025-12-19,2026-07-31


In [8]:
query_maestro_lote = f"""
WITH base AS (
    SELECT
        ac.Campaña,
        v.Variedad,
        f.Fundo,
        p.Parcela,
        sp.SubParcela AS SubParcela_Origen,
        CASE WHEN sp.SubParcela = 'P19C' THEN 'P19' ELSE sp.SubParcela END AS SubParcela,
        l.Nombre AS Lote,
        sp.PodaEstacion,
        v.TipoCultivo,

        CAST(
            CASE
                WHEN sp_hist.idSubParcela IS NOT NULL THEN sp_hist.FechaIniCos
                ELSE sp.FechaIniCos
            END AS DATE
        ) AS FechaIniCos,

        CAST(
            CASE
                WHEN sp_hist.idSubParcela IS NOT NULL THEN sp_hist.FechaFinCos
                ELSE sp.FechaFinCos
            END AS DATE
        ) AS FechaFinCos,

        l.Densidad,
        ac.Superficie
    FROM AreaCultivable ac
        INNER JOIN Lote l ON l.idLote = ac.idLote
        -- SubParcela actualmente asociada al lote
        INNER JOIN SubParcela sp ON sp.idSubParcela = l.idSubParcela
        -- Versión histórica de la SubParcela correspondiente
        -- a la campaña y variedad de AreaCultivable
        LEFT JOIN SubParcela sp_hist
            ON sp_hist.idParcela = sp.idParcela
            AND sp_hist.SubParcela = sp.SubParcela
            AND sp_hist.Campania = ac.Campaña
            AND sp_hist.idVariedad = ac.idVariedad
        INNER JOIN Parcela p ON p.id = sp.idParcela
        INNER JOIN Fundo f ON f.id = p.IdFundo
        INNER JOIN Variedad v ON v.idVariedad = ac.idVariedad
    WHERE 1 = 1
        AND sp.Estado = 1
        -- AND ac.Estado = 1
        AND ac.Campaña >= {CAMPANIA - 1} -- traer una campaña adicional
        AND v.idCultivo = 9
)

SELECT
    Campaña,
    Variedad,
    Fundo,
    Parcela,
    SubParcela_Origen,
    SubParcela,
    Lote,
    PodaEstacion,
    TipoCultivo,
    FechaIniCos,
    FechaFinCos,
    SUM(Densidad) AS Nro_Plantas,
    SUM(Superficie) AS Ha,
    CAST(SUM(Densidad) AS FLOAT) / NULLIF(CAST(SUM(Superficie) AS FLOAT), 0) AS Densidad_Plantas
FROM base
GROUP BY
    Campaña,
    Variedad,
    Fundo,
    Parcela,
    SubParcela_Origen,
    SubParcela,
    Lote,
    PodaEstacion,
    TipoCultivo,
    FechaIniCos,
    FechaFinCos
"""

with engine_proy.begin() as conn:
    df_maestro_lote = (
        pd.read_sql(query_maestro_lote, conn)
        .assign(
            Campaña = lambda d: d['Campaña'].astype('int64'),
            FechaIniCos=lambda d: pd.to_datetime(d['FechaIniCos']),
            FechaFinCos=lambda d: pd.to_datetime(d['FechaFinCos']),
            PodaEstacion=lambda d: d['PodaEstacion'].mask( d['SubParcela'].eq('P19') & d['Lote'].eq('L593'), 'Verano' ),
            Variedad=lambda d: d['Variedad']
                .mask( d['SubParcela'].isin(['P77', 'P75']), 'Ventura' )
                .mask( d['SubParcela'].eq('P20C') & d['Campaña'].isin([2020, 2021, 2022]) & d['Variedad'].eq('Ventura'), 'Biloxi')
                .mask( d['SubParcela'].eq('P302A') & d['Campaña'].isin([2022]) & d['Variedad'].eq('Malibu'), 'Madeira'),
            TipoCultivo=lambda d: d['TipoCultivo'].mask( d['SubParcela'].eq('P20C'), 'Organico' ),
        )

        # Correcciones temporales de registros históricos incorrectos
        .loc[lambda d: ~(
            (d["Campaña"].eq(2020) & d["Lote"].isin(["L1365B", "L1365C"]) & d["FechaIniCos"].eq("2021-08-02")) |
            (d["Campaña"].eq(2020) & d["Lote"].isin(["L3365", "L3370"]) & d["FechaIniCos"].eq("2021-05-31")) |
            (d["Campaña"].eq(2020) & d["Lote"].isin(["L4426", "L4431", "L4437", "L4444"]) & d["FechaIniCos"].eq("2021-07-16"))
        )]
        .assign(
            FechaIniCos=lambda d: d["FechaIniCos"].mask(
                d["Campaña"].eq(2020) & d["Lote"].eq("L2401") &
                d["SubParcela"].eq("P121") & d["Variedad"].eq("Arana"),
                pd.Timestamp("2020-07-09")
            ),
            FechaFinCos=lambda d: d["FechaFinCos"].mask(
                d["Campaña"].eq(2020) & d["Lote"].eq("L2401") &
                d["SubParcela"].eq("P121") & d["Variedad"].eq("Arana"),
                pd.Timestamp("2021-01-21")
            ),
        )

        # excluir registros históricos incorrectos
        .query("~(Campaña == 2020 & Lote == 'L2397' & SubParcela == 'P121A')")
        .query("~(Campaña == 2020 & Lote == 'L2401' & SubParcela == 'P121A')")

        .merge(
            df_maestro_fechas.rename(columns={"SubParcela": "SubParcela_Origen"}),
            on=["Campaña", "SubParcela_Origen"],
            how="left",
            validate="many_to_one"
        )
        .assign(
            FechaIniCos=lambda d: d["FechaIniCos_M"].combine_first(d["FechaIniCos"]),
            FechaFinCos=lambda d: d["FechaFinCos_M"].combine_first(d["FechaFinCos"]),
        )
        .drop(columns=["FechaIniCos_M", "FechaFinCos_M"])

        .groupby(['Campaña', 'Variedad', 'Fundo', 'Parcela', 'SubParcela', 'Lote', 'PodaEstacion', 'TipoCultivo', 'FechaIniCos', 'FechaFinCos'], as_index=False, dropna=False)
        .agg({
            'Nro_Plantas': 'max',
            'Ha': 'sum'
        })
        # Exclusión del registro específico porque está duplicado
        .query("~(Campaña == 2025 & SubParcela == 'P121' & Lote == 'L2393')")
        # Importante para que shift tome correctamente la campaña anterior
        .sort_values(["Fundo", "Parcela", "Lote", "Campaña"])
        .assign(
            FechaInicioCampaña=lambda d:
                d.groupby(["Fundo", "Parcela", "Lote"], dropna=False)["FechaFinCos"]
                .shift(1).add(pd.Timedelta(days=1))
                .fillna(
                    pd.concat([
                        d["FechaIniCos"],
                        pd.to_datetime(d["Campaña"].astype(str) + "-07-01")
                    ], axis=1).min(axis=1)
                    .where(d["FechaIniCos"].notna() & d["FechaFinCos"].notna())
                ),

            Nro_Plantas        = lambda d: d['Nro_Plantas'].fillna( d.groupby(['Fundo', 'Parcela', 'SubParcela', 'Lote'], dropna=False)['Nro_Plantas'].bfill() ),
            Densidad_Plantas   = lambda d: d['Nro_Plantas'] / d['Ha'],
            Duracion_Cosecha   = lambda d: (d["FechaFinCos"] - d["FechaIniCos"]).dt.days + 1,
        )
        # Ajustar casos donde el inicio calculado cae después del inicio real de cosecha
        .assign(
            FechaInicioCampaña=lambda d: d["FechaInicioCampaña"].mask(
                d["FechaInicioCampaña"] > d["FechaIniCos"],
                d["FechaIniCos"]
            )
        )
        # CAMPANIA-1 solo era auxiliar para calcular FechaInicioCampaña
        .loc[lambda d: d["Campaña"] >= CAMPANIA]
        .reset_index(drop=True)
    )

validar_unicidad( df_maestro_lote, ['Campaña', 'Lote'], 'df_maestro_lote' )

assert df_maestro_lote["Duracion_Cosecha"].dropna().ge(1).all(), \
    "Hay registros con FechaFinCos anterior a FechaIniCos"

df_maestro_lote.head()

,Campaña,Variedad,Fundo,Parcela,SubParcela,Lote,PodaEstacion,TipoCultivo,FechaIniCos,FechaFinCos,Nro_Plantas,Ha,FechaInicioCampaña,Densidad_Plantas,Duracion_Cosecha
0,2021,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2021-06-04,2021-11-11,3457.0,0.5,2020-12-15,6914.0,161.0
1,2022,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2022-04-28,2022-11-11,3457.0,0.5,2021-11-12,6914.0,198.0
2,2023,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2023-04-21,2024-02-01,3457.0,0.5,2022-11-12,6914.0,287.0
3,2024,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2024-08-03,2025-02-14,3457.0,0.5,2024-02-02,6914.0,196.0
4,2025,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2025-08-05,2026-02-28,3457.0,0.5,2025-02-15,6914.0,208.0


In [28]:
# df_maestro_lote[(df_maestro_lote['Campaña'] == 2020) & (df_maestro_lote['SubParcela'] == 'P20C')][['FechaIniCos', 'FechaFinCos']].drop_duplicates()
# df_maestro_lote[(df_maestro_lote['Campaña'] == 2020) & (df_maestro_lote['SubParcela'] == 'P20C') & (df_maestro_lote['FechaIniCos']== '2020-07-15')]

In [ ]:
# detectar automáticamente campañas cuyo supuesto inicio calculado cae después del inicio de cosecha.
registros_FechaInicioCampaña_mayor_a_FechaIniCos = (
    df_maestro_lote.loc[df_maestro_lote['FechaInicioCampaña'] > df_maestro_lote['FechaIniCos']]
)

assert registros_FechaInicioCampaña_mayor_a_FechaIniCos.empty, \
    f"Hay {len(registros_FechaInicioCampaña_mayor_a_FechaIniCos):,} registros con FechaInicioCampaña > FechaIniCos"

# 2. Define la ruta usando el patrón pathlib
ruta_archivo = BASE / "9. ANALISTA_PROYECCION" / "6. ENRIQUE_MOSQUEIRA" / "projects" / "registros_FechaInicioCampaña_mayor_a_FechaIniCos.csv"

# 3. Exporta a Excel
registros_FechaInicioCampaña_mayor_a_FechaIniCos.to_csv(ruta_archivo, index=False)

In [10]:
df_historico_poda = (
    pd.read_excel(ruta_historico_poda, sheet_name = 'Hoja1')
    .drop(columns   = ['Poda', 'Variedad', 'TipoCampo', 'Fundo', 'SubFundo', 'Área (ha)', 'Plantas'])
    .melt(
        id_vars    = ['SubParcela'],
        var_name   = 'Campaña',
        value_name = 'Periodo_Poda'
    )
    .dropna(subset=['Periodo_Poda'])
    .assign(
        Campaña=lambda d: d['Campaña'].str.replace('Poda C', '20').astype(int),
        Periodo_Poda=lambda d: d['Periodo_Poda'].astype(int),
        Anio_Poda=lambda d: d['Periodo_Poda'].astype(int).astype(str).str[:4].astype(int),
        Semana_Poda=lambda d: d['Periodo_Poda'].astype(int).astype(str).str[-2:].astype(int)
    )
    .loc[lambda d: d['Campaña'] >= CAMPANIA]
)

validar_unicidad( df_historico_poda, ["Campaña", "SubParcela"], "df_historico_poda" )

df_historico_poda.head()

,SubParcela,Campaña,Periodo_Poda,Anio_Poda,Semana_Poda
0,P62,2021,202102,2021,2
1,P64,2021,202103,2021,3
2,P66A,2021,202102,2021,2
3,P67,2021,202102,2021,2
4,P69A,2021,202102,2021,2


In [11]:
df_maestro_lote_poda = (
    pd.merge(
        df_maestro_lote,
        df_historico_poda,
        on=["Campaña", "SubParcela"],
        how="left"
    )
    .assign(
        # Nº de campaña observada del lote (proxy de edad): 1, 2, 3, ...
        Numero_Campaña_Lote=lambda d: d.groupby("Lote")["Campaña"].rank(method="dense").astype("int64"),
        # Alternativa que respeta saltos de campaña (más cercano a "años"):
        Años_Desde_1ra_Campaña=lambda d: d["Campaña"] - d.groupby("Lote")["Campaña"].transform("min") + 1,
    )
)

validar_unicidad( df_maestro_lote_poda, ["Campaña", "Lote"], "df_maestro_lote_poda" )
validar_cruce(df_maestro_lote, df_maestro_lote_poda, "Maestro x Poda")

df_maestro_lote_poda.head()

✅ [Maestro x Poda] Validación exitosa: El número de filas se mantuvo intacto en 14230.


,Campaña,Variedad,Fundo,Parcela,SubParcela,Lote,PodaEstacion,TipoCultivo,FechaIniCos,FechaFinCos,Nro_Plantas,Ha,FechaInicioCampaña,Densidad_Plantas,Duracion_Cosecha,Periodo_Poda,Anio_Poda,Semana_Poda,Numero_Campaña_Lote,Años_Desde_1ra_Campaña
0,2021,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2021-06-04,2021-11-11,3457.0,0.5,2020-12-15,6914.0,161.0,202051.0,2020.0,51.0,1,1
1,2022,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2022-04-28,2022-11-11,3457.0,0.5,2021-11-12,6914.0,198.0,202145.0,2021.0,45.0,2,2
2,2023,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2023-04-21,2024-02-01,3457.0,0.5,2022-11-12,6914.0,287.0,202245.0,2022.0,45.0,3,3
3,2024,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2024-08-03,2025-02-14,3457.0,0.5,2024-02-02,6914.0,196.0,202405.0,2024.0,5.0,4,4
4,2025,Ventura,AGRICULTOR,P61,P61,L3376,Verano,Convencional,2025-08-05,2026-02-28,3457.0,0.5,2025-02-15,6914.0,208.0,202508.0,2025.0,8.0,5,5


In [12]:
# Celda para validar FechaIniCos, FechaFinCos y poda
# CONTROL AGRONÓMICO: Campaña + SubParcela

_cal_poda = (
    df_calendario_juliano[["Año", "Semana", "FechaInicioSemana"]]
    .drop_duplicates(["Año", "Semana"])
    .rename(columns={"Año": "Anio_Poda", "Semana": "Semana_Poda", "FechaInicioSemana": "Fecha_Poda"})
)

def _observacion(r):
    obs = []
    if r["Faltan_FechaIniCos"] > 0: obs.append("Falta FechaIniCos")
    if r["Faltan_FechaFinCos"] > 0: obs.append("Falta FechaFinCos")
    if r["Rangos_Invalidos"] > 0: obs.append("FechaFinCos < FechaIniCos")
    if r["N_FechaIniCos"] > 1: obs.append("Múltiples FechaIniCos")
    if r["N_FechaFinCos"] > 1: obs.append("Múltiples FechaFinCos")
    if r["N_Periodo_Poda"] > 1: obs.append("Múltiples Periodo_Poda")
    if pd.isna(r["Periodo_Poda"]): obs.append("Falta poda")
    elif pd.isna(r["Fecha_Poda"]): obs.append("Poda fuera calendario")
    if r["Poda_Despues_Cosecha"] > 0: obs.append("Poda posterior a FechaIniCos")
    return " | ".join(obs) if obs else "OK"

control_agronomico = (
    df_maestro_lote_poda
    .assign(
        FechaIniCos=lambda d: pd.to_datetime(d["FechaIniCos"], errors="coerce"),
        FechaFinCos=lambda d: pd.to_datetime(d["FechaFinCos"], errors="coerce")
    )
    .merge(_cal_poda, on=["Anio_Poda", "Semana_Poda"], how="left", validate="many_to_one")
    .assign(
        Rango_Invalido=lambda d: d["FechaFinCos"].lt(d["FechaIniCos"]),
        Poda_Despues_Cosecha=lambda d: d["Fecha_Poda"].gt(d["FechaIniCos"])
    )
    .groupby(["Campaña", "SubParcela"], as_index=False)
    .agg(
        Lotes=("Lote", "nunique"),
        FechaIniCos_Min=("FechaIniCos", "min"), FechaIniCos_Max=("FechaIniCos", "max"), N_FechaIniCos=("FechaIniCos", "nunique"),
        FechaFinCos_Min=("FechaFinCos", "min"), FechaFinCos_Max=("FechaFinCos", "max"), N_FechaFinCos=("FechaFinCos", "nunique"),
        Periodo_Poda=("Periodo_Poda", "first"), N_Periodo_Poda=("Periodo_Poda", "nunique"),
        Anio_Poda=("Anio_Poda", "first"), Semana_Poda=("Semana_Poda", "first"), Fecha_Poda=("Fecha_Poda", "first"),
        Faltan_FechaIniCos=("FechaIniCos", lambda s: s.isna().sum()),
        Faltan_FechaFinCos=("FechaFinCos", lambda s: s.isna().sum()),
        Rangos_Invalidos=("Rango_Invalido", "sum"),
        Poda_Despues_Cosecha=("Poda_Despues_Cosecha", "sum")
    )
    .assign(
        Dias_Poda_a_IniCos=lambda d: (d["FechaIniCos_Min"] - d["Fecha_Poda"]).dt.days,
        Dias_Ventana_Cosecha=lambda d: (d["FechaFinCos_Max"] - d["FechaIniCos_Min"]).dt.days
    )
)

control_agronomico["Observacion"] = control_agronomico.apply(_observacion, axis=1)

control_agronomico_revisar = (
    control_agronomico
    .loc[lambda d: d["Observacion"].ne("OK")]
    .sort_values(["Campaña", "SubParcela"])
    .reset_index(drop=True)
)

print(f"SubParcelas revisadas: {len(control_agronomico):,}")
print(f"SubParcelas con observaciones: {len(control_agronomico_revisar):,}")

control_agronomico_revisar

SubParcelas revisadas: 548
SubParcelas con observaciones: 175


,Campaña,SubParcela,Lotes,FechaIniCos_Min,FechaIniCos_Max,N_FechaIniCos,FechaFinCos_Min,FechaFinCos_Max,N_FechaFinCos,Periodo_Poda,N_Periodo_Poda,Anio_Poda,Semana_Poda,Fecha_Poda,Faltan_FechaIniCos,Faltan_FechaFinCos,Rangos_Invalidos,Poda_Despues_Cosecha,Dias_Poda_a_IniCos,Dias_Ventana_Cosecha,Observacion
0,2021,P121,45,2021-07-24,2021-07-24,1,2022-03-05,2022-03-05,1,NaN,0,NaN,NaN,NaT,0,0,0,0,NaN,224.0,Falta poda
1,2021,P153B,38,2021-06-10,2021-06-10,1,2021-11-27,2021-11-27,1,NaN,0,NaN,NaN,NaT,0,0,0,0,NaN,170.0,Falta poda
2,2021,P163,30,2021-06-21,2021-06-21,1,2021-11-27,2021-11-27,1,202153.0,1,2021.0,53.0,NaT,0,0,0,0,NaN,159.0,Poda fuera calendario
3,2021,P19,55,2021-07-31,2021-12-13,2,2022-01-04,2022-06-09,2,202104.0,1,2021.0,4.0,2021-01-25,0,0,0,0,187.0,313.0,Múltiples FechaIniCos | Múltiples FechaFinCos
4,2021,P57A,18,2021-08-04,2021-08-04,1,2021-12-20,2021-12-20,1,NaN,0,NaN,NaN,NaT,0,0,0,0,NaN,138.0,Falta poda
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,2026,P69B,29,2025-12-07,2025-12-07,1,2027-01-06,2027-01-06,1,202551.0,1,2025.0,51.0,2025-12-15,0,0,0,29,-8.0,395.0,Poda posterior a FechaIniCos
171,2026,P75,32,2026-03-01,2026-03-01,1,2027-03-31,2027-03-31,1,202610.0,1,2026.0,10.0,2026-03-02,0,0,0,32,-1.0,395.0,Poda posterior a FechaIniCos
172,2026,P79,58,2025-11-29,2025-11-29,1,2026-12-29,2026-12-29,1,202549.0,1,2025.0,49.0,2025-12-01,0,0,0,58,-2.0,395.0,Poda posterior a FechaIniCos
173,2026,P80A,47,2025-11-28,2025-11-28,1,2026-12-28,2026-12-28,1,202549.0,1,2025.0,49.0,2025-12-01,0,0,0,47,-3.0,395.0,Poda posterior a FechaIniCos


In [25]:
ruta_archivo2 = ( BASE / "9. ANALISTA_PROYECCION" / "6. ENRIQUE_MOSQUEIRA" / "projects" / "control_agronomico_revisar.csv" )

control_agronomico_revisar.to_csv(ruta_archivo2, index=False)

In [13]:
# df_maestro_semanal — Grano IN: Campaña+Lote | Grano OUT: Campaña+Lote+AñoSemana (semanas continuas)
# Frontera: FechaInicioCampaña -> FechaFinCos | Fecha->Semana: merge equi sobre calendario diario (único por Fecha)

# 1. Maestro con fronteras y semana de poda
_df = (
    df_maestro_lote_poda
    .assign(
        FechaInicioCampaña=lambda d: pd.to_datetime(d["FechaInicioCampaña"]).dt.normalize(),
        FechaFinCos=lambda d: pd.to_datetime(d["FechaFinCos"]).dt.normalize(),
    )
    # corr_ini / corr_fin
    .merge(
        df_calendario_juliano[["Fecha", "Semana_Correlativa"]]
        .rename(columns={"Fecha": "FechaInicioCampaña", "Semana_Correlativa": "corr_ini"}),
        on="FechaInicioCampaña", how="left", validate="many_to_one"
    )
    .merge(
        df_calendario_juliano[["Fecha", "Semana_Correlativa"]]
        .rename(columns={"Fecha": "FechaFinCos", "Semana_Correlativa": "corr_fin"}),
        on="FechaFinCos", how="left", validate="many_to_one"
    )
    # corr_poda: (Anio_Poda, Semana_Poda) -> Semana_Correlativa
    .merge(
        df_calendario_juliano[["Año", "Semana", "Semana_Correlativa"]]
        .drop_duplicates(["Año", "Semana"])
        .rename(columns={
            "Año": "Anio_Poda",
            "Semana": "Semana_Poda",
            "Semana_Correlativa": "Semana_Correlativa_Poda"
        }),
        on=["Anio_Poda", "Semana_Poda"], how="left", validate="many_to_one"
    )
)

# 2. Exportar registros excluidos por frontera inválida / fuera de calendario
registros_frontera_invalida = (
    _df.loc[ _df[["corr_ini", "corr_fin"]].isna().any(axis=1) | (_df["corr_fin"] < _df["corr_ini"]) ]
    .assign(
        Motivo=lambda d: np.select(
            [
                d["corr_ini"].isna() & d["corr_fin"].isna(),
                d["corr_ini"].isna(),
                d["corr_fin"].isna(),
                d["corr_fin"] < d["corr_ini"],
            ],
            [
                "Inicio y fin fuera calendario",
                "Inicio fuera calendario",
                "Fin fuera calendario",
                "Fin anterior al inicio",
            ],
            default="Rango inválido"
        )
    )
)

ruta_archivo = ( BASE / "9. ANALISTA_PROYECCION" / "6. ENRIQUE_MOSQUEIRA" / "projects" / "registros_frontera_invalida.csv" )

registros_frontera_invalida.to_csv(ruta_archivo, index=False)

# 3. Expandir a semanas continuas
df_maestro_semanal = (
    _df
    # conservar solo fronteras válidas
    .loc[lambda d: d[["corr_ini", "corr_fin"]].notna().all(axis=1) & (d["corr_fin"] >= d["corr_ini"])]
    # expandir [corr_ini, corr_fin]
    .assign( Semana_Correlativa=lambda d: [ np.arange(a, b + 1) for a, b in zip(d["corr_ini"], d["corr_fin"]) ] )
    .explode("Semana_Correlativa", ignore_index=True)
    .astype({"Semana_Correlativa": "int64"})
    # etiquetas semanales
    .merge(
        df_calendario_juliano[["Semana_Correlativa", "Año", "Semana", "AñoSemana", "FechaInicioSemana", "FechaFinSemana"]]
        .drop_duplicates("Semana_Correlativa"),
        on="Semana_Correlativa",
        how="left",
        validate="many_to_one"
    )
    .assign( Semanas_Desde_Poda=lambda d: d["Semana_Correlativa"] - d["Semana_Correlativa_Poda"] )
    .drop(columns=["corr_ini", "corr_fin"])
    .sort_values(["Campaña", "Lote", "Semana_Correlativa"])
    .reset_index(drop=True)
)

# 4. Validaciones
validar_unicidad(
    df_maestro_semanal,
    ["Campaña", "Lote", "AñoSemana"],
    "df_maestro_semanal"
)

assert (
    _cont := df_maestro_semanal
    .groupby(["Campaña", "Lote"])["Semana_Correlativa"]
    .apply(lambda s: s.diff().dropna().eq(1).all())
).all(), (
    f"Hay {(~_cont).sum():,} lote-campaña con semanas no continuas. "
    f"Primeros casos: {_cont[~_cont].index.tolist()[:5]}"
)

# 5. Resumen
print(
    f"⚠️ Excluidos por frontera inválida/fuera calendario: {len(registros_frontera_invalida):,} | "
    f"archivo: {ruta_archivo.name}"
)

print(
    f"✅ df_maestro_semanal: {len(df_maestro_semanal):,} filas | "
    f"{df_maestro_semanal[['Campaña','Lote']].drop_duplicates().shape[0]:,} lote-campaña | "
    f"sin poda: "
    f"{df_maestro_semanal.loc[df_maestro_semanal['Semana_Correlativa_Poda'].isna(), ['Campaña','Lote']].drop_duplicates().shape[0]:,} | "
    f"continuidad: OK"
)

df_maestro_semanal.head()

⚠️ Excluidos por frontera inválida/fuera calendario: 119 | archivo: registros_frontera_invalida.csv
✅ df_maestro_semanal: 792,068 filas | 14,111 lote-campaña | sin poda: 849 | continuidad: OK


,Campaña,Variedad,Fundo,Parcela,SubParcela,Lote,PodaEstacion,TipoCultivo,FechaIniCos,FechaFinCos,Nro_Plantas,Ha,FechaInicioCampaña,Densidad_Plantas,Duracion_Cosecha,Periodo_Poda,Anio_Poda,Semana_Poda,Numero_Campaña_Lote,Años_Desde_1ra_Campaña,Semana_Correlativa_Poda,Semana_Correlativa,Año,Semana,AñoSemana,FechaInicioSemana,FechaFinSemana,Semanas_Desde_Poda
0,2021,Biloxi,GLORIA,P18B,P18B,L1206,Verano,Convencional,2021-07-30,2022-01-15,2610.0,0.43,2021-02-04,6069.767442,170.0,202105.0,2021.0,5.0,1,1,59.0,59,2021,5,2106,2021-02-01,2021-02-07,0.0
1,2021,Biloxi,GLORIA,P18B,P18B,L1206,Verano,Convencional,2021-07-30,2022-01-15,2610.0,0.43,2021-02-04,6069.767442,170.0,202105.0,2021.0,5.0,1,1,59.0,60,2021,6,2107,2021-02-08,2021-02-14,1.0
2,2021,Biloxi,GLORIA,P18B,P18B,L1206,Verano,Convencional,2021-07-30,2022-01-15,2610.0,0.43,2021-02-04,6069.767442,170.0,202105.0,2021.0,5.0,1,1,59.0,61,2021,7,2108,2021-02-15,2021-02-21,2.0
3,2021,Biloxi,GLORIA,P18B,P18B,L1206,Verano,Convencional,2021-07-30,2022-01-15,2610.0,0.43,2021-02-04,6069.767442,170.0,202105.0,2021.0,5.0,1,1,59.0,62,2021,8,2109,2021-02-22,2021-02-28,3.0
4,2021,Biloxi,GLORIA,P18B,P18B,L1206,Verano,Convencional,2021-07-30,2022-01-15,2610.0,0.43,2021-02-04,6069.767442,170.0,202105.0,2021.0,5.0,1,1,59.0,63,2021,9,2110,2021-03-01,2021-03-07,4.0


In [14]:
# ------------------------------------------------------------------
# Parámetros agronómicos / operativos (única fuente de verdad)
# ------------------------------------------------------------------
TEMP_BASE_GDD: float = 10.0    # °C, base para GDD | Para arándano lo típico es ~7 °C
VPD_ALTO: float = 1.20         # kPa
VPD_MUY_ALTO: float = 1.50     # kPa
RAD_ALTA: float = 500.0        # W/m2
REGISTROS_ESPERADOS: float = 48.0  # 48 registros de 30 min por día
COBERTURA_MINIMA: float = 0.80 # umbral para marcar el día como válido

query_clima = f"""
-- df_clima
-- Granularidad: Fundo + Fecha
-- Fuente: registros cada 30 minutos
WITH clima_base AS (
    SELECT
        CASE
            WHEN tt.Fundo = 'Agricultor 2' THEN 'AGRICULTOR'
            WHEN tt.Fundo = 'Agro Alegre' THEN 'AGROALEGRE'
            WHEN tt.Fundo LIKE 'Yakuy Minka %' THEN 'YAKUY MINKA'
            ELSE UPPER(LTRIM(RTRIM(tt.Fundo)))
        END                                 AS Fundo,
        CAST(tt.Fecha AS DATE)              AS Fecha,
        -- TRY_CONVERT(TIME(0), tt.Hora)    AS Hora,
        TRY_CAST(tt.Tem_Externa AS FLOAT)   AS Tem_Externa,
        TRY_CAST(tt.Tem_Maxima AS FLOAT)    AS Tem_Maxima,
        TRY_CAST(tt.Tem_Minima AS FLOAT)    AS Tem_Minima,
        TRY_CAST(tt.Hum_Relativa AS FLOAT)  AS Hum_Relativa,
        TRY_CAST(tt.vpd_valor AS FLOAT)     AS VPD,
        TRY_CAST(tt.Rad_Solar AS FLOAT)     AS Rad_Solar,
        TRY_CAST(tt.Lluvia AS FLOAT)        AS Lluvia,
        TRY_CAST(tt.Vel_DelViento AS FLOAT) AS Vel_Viento,
        TRY_CAST(tt.Max_DelViento AS FLOAT) AS Vel_Viento_Max,
        TRY_CAST(tt.Eva_Et AS FLOAT)        AS Eva_ET,
        TRY_CAST(tt.Eto_Promedio AS FLOAT)  AS ETo,
        TRY_CAST(tt.Pun_Rocio AS FLOAT)     AS Punto_Rocio
    FROM TablaTemperatura tt
    WHERE tt.Fecha >= '{FECHA_INICIO}'
),
-- Percentiles calculados a nivel Fundo + Fecha
BasePercentiles AS (
    SELECT
        *,
        PERCENTILE_CONT(0.90)
        WITHIN GROUP (ORDER BY VPD)
        OVER (PARTITION BY Fundo, Fecha) AS VPD_P90_Dia,
        PERCENTILE_CONT(0.90)
        WITHIN GROUP (ORDER BY Vel_Viento)
        OVER (PARTITION BY Fundo, Fecha) AS Viento_P90_Dia
    FROM clima_base
),
clima_diario AS (
    SELECT
        Fundo,
        Fecha,

        -- CALIDAD / COBERTURA DE INFORMACION
        COUNT(*) AS Registros_Observados,
        COUNT(Tem_Externa) AS Registros_Temperatura,
        COUNT(*) / {REGISTROS_ESPERADOS} AS Cobertura_Registros,
        CASE
            WHEN COUNT(Tem_Externa) >= {REGISTROS_ESPERADOS} * {COBERTURA_MINIMA} THEN 1
            ELSE 0
        END AS Dia_Valido,
        COUNT(Tem_Externa) / {REGISTROS_ESPERADOS} AS Cobertura_Temperatura,

        -- TEMPERATURA
        AVG(Tem_Externa)                          AS Temp_Media,
        MAX(COALESCE(Tem_Maxima, Tem_Externa))    AS Temp_Max,
        MIN(COALESCE(Tem_Minima, Tem_Externa))    AS Temp_Min,
        STDEV(Tem_Externa)                        AS Temp_DesvStd,
        -- Temperatura durante horas con radiacion solar
        AVG(CASE WHEN Rad_Solar > 5 THEN Tem_Externa END) AS Temp_Media_Dia,
        AVG(CASE WHEN ISNULL(Rad_Solar, 0) <= 5 THEN Tem_Externa END) AS Temp_Media_Noche,
        -- Horas de exposición. Cada registro = 30 minutos = 0.5 horas
        SUM(CASE WHEN Tem_Externa > 28 THEN 0.5 ELSE 0 END) AS Horas_Temp_Mayor_28,
        SUM(CASE WHEN Tem_Externa > 30 THEN 0.5 ELSE 0 END) AS Horas_Temp_Mayor_30,
        SUM(CASE WHEN Tem_Externa > 32 THEN 0.5 ELSE 0 END) AS Horas_Temp_Mayor_32,
        SUM(CASE WHEN Tem_Externa > 35 THEN 0.5 ELSE 0 END) AS Horas_Temp_Mayor_35,
        SUM(CASE WHEN Tem_Externa < 15 THEN 0.5 ELSE 0 END) AS Horas_Temp_Menor_15,
        SUM(CASE WHEN Tem_Externa < 10 THEN 0.5 ELSE 0 END) AS Horas_Temp_Menor_10,

        -- HUMEDAD RELATIVA
        AVG(Hum_Relativa)    AS HR_Media,
        MIN(Hum_Relativa)    AS HR_Min,
        MAX(Hum_Relativa)    AS HR_Max,
        STDEV(Hum_Relativa)  AS HR_DesvStd,
        AVG(CASE WHEN Rad_Solar > 5 THEN Hum_Relativa END)             AS HR_Media_Dia,
        AVG(CASE WHEN ISNULL(Rad_Solar, 0) <= 5 THEN Hum_Relativa END) AS HR_Media_Noche,
        SUM(CASE WHEN Hum_Relativa > 80 THEN 0.5 ELSE 0 END) AS Horas_HR_Mayor_80,
        SUM(CASE WHEN Hum_Relativa > 85 THEN 0.5 ELSE 0 END) AS Horas_HR_Mayor_85,
        SUM(CASE WHEN Hum_Relativa > 90 THEN 0.5 ELSE 0 END) AS Horas_HR_Mayor_90,
        SUM(CASE WHEN Hum_Relativa > 95 THEN 0.5 ELSE 0 END) AS Horas_HR_Mayor_95,

        -- VPD
        AVG(VPD)                                  AS VPD_Medio,
        MIN(VPD)                                  AS VPD_Min,
        MAX(VPD)                                  AS VPD_Max,
        STDEV(VPD) AS VPD_DesvStd,
        AVG(CASE WHEN Rad_Solar > 5 THEN VPD END)           AS VPD_Medio_Dia,
        AVG(CASE WHEN ISNULL(Rad_Solar, 0) <= 5 THEN VPD END) AS VPD_Medio_Noche,
        SUM(CASE WHEN VPD > {VPD_ALTO}     THEN 0.5 ELSE 0 END) AS Horas_VPD_Alto,
        SUM(CASE WHEN VPD > {VPD_MUY_ALTO} THEN 0.5 ELSE 0 END) AS Horas_VPD_MuyAlto,
        MAX(VPD_P90_Dia) AS VPD_P90,

        -- RADIACIÓN SOLAR
        AVG(Rad_Solar)                                  AS Rad_Solar_Media,
        AVG(CASE WHEN Rad_Solar > 5 THEN Rad_Solar END) AS Radiacion_Media_Diurna,
        MAX(Rad_Solar)                                  AS Rad_Solar_Max,
        -- W/m2 -> MJ/m2 (intervalo = 1800 s)
        AVG(Rad_Solar) * 0.0864 AS Radiacion_MJ_m2,   -- media diaria (W/m2) -> MJ/m2/día
        SUM(CASE WHEN Rad_Solar > {RAD_ALTA} THEN 0.5 ELSE 0 END) AS Horas_Radiacion_Alta,
        -- Horas de luz (proxy de fotoperiodo observado)
        SUM(CASE WHEN Rad_Solar > 5 THEN 0.5 ELSE 0 END) AS Horas_Luz,

        -- LLUVIA
        SUM(Lluvia)                               AS Lluvia_Total,
        MAX(Lluvia)                               AS Lluvia_Max,
        CASE
            WHEN SUM(ISNULL(Lluvia, 0)) > 0 THEN 1
            ELSE 0
        END AS Dia_Con_Lluvia,
        SUM(CASE WHEN Lluvia > 0 THEN 0.5 ELSE 0 END) AS Horas_Con_Lluvia,

        -- VIENTO
        AVG(Vel_Viento)                           AS Vel_Viento_Media,
        MAX(COALESCE(Vel_Viento_Max, Vel_Viento)) AS Vel_Viento_Max,
        MAX(Viento_P90_Dia) AS Viento_P90,

        -- EVAPOTRANSPIRACIÓN
        SUM(Eva_ET)                               AS Eva_ET_Total, -- parece incremental

        -- PUNTO DE ROCÍO
        AVG(Punto_Rocio)                          AS Punto_Rocio_Medio,
        MIN(Punto_Rocio)                          AS Punto_Rocio_Min,
        MAX(Punto_Rocio)                          AS Punto_Rocio_Max

    FROM BasePercentiles
    GROUP BY
        Fundo,
        Fecha
)
-- VARIABLES DERIVADAS FINALES (todo en SQL)
SELECT
    *,
    -- Temperatura media desde extremos
    (Temp_Max + Temp_Min) / 2.0            AS Temp_Media_MaxMin,
    -- Amplitud térmica
    Temp_Max - Temp_Min                    AS Amplitud_Termica,
    -- Growing Degree Days (base {TEMP_BASE_GDD} °C)
    CASE
        WHEN ( ((Temp_Max + Temp_Min) / 2.0) - {TEMP_BASE_GDD} ) > 0
        THEN ( ((Temp_Max + Temp_Min) / 2.0) - {TEMP_BASE_GDD} )
        ELSE 0
    END                                    AS GDD_Diario,
    -- Diferencia temperatura día/noche
    Temp_Media_Dia - Temp_Media_Noche      AS Diferencia_Temp_Dia_Noche,
    -- Diferencia HR noche/día
    HR_Media_Noche - HR_Media_Dia          AS Diferencia_HR_Noche_Dia,
    -- Diferencia temperatura - punto de rocío
    Temp_Media - Punto_Rocio_Medio         AS Delta_Temp_Punto_Rocio
FROM clima_diario
ORDER BY Fundo, Fecha;
"""

# Latitud por fundo (°, hemisferio sur = negativo). Fuente: coordenadas de cada fundo.
LAT_POR_FUNDO: dict[str, float] = {
    "AGRICULTOR":    -8.5619254,
    "AGROALEGRE":    -4.8334190,
    "AGROMAS":       -8.4308302,
    "FRUSOL I":      -8.4743988,
    "GLORIA":        -8.4674215,
    "MAR VERDE":     -8.4584382,
    "ORO AZUL":      -8.5060860,
    "SAN JOSE":      -8.4284807,
    "SINCROMAX":     -8.5207130,
    "TERRA":         -5.0111210,
    "YAKUY MINKA":   -8.5342668,
}

LAT_DEFAULT: float = -8.5   # VALLE PERDIDO no tiene coordenada -> usa el clúster dominante (Virú/La Libertad)

def eto_hargreaves(
    temp_max: pd.Series,
    temp_min: pd.Series,
    temp_media: pd.Series,
    fecha: pd.Series,
    lat_deg: pd.Series,
) -> pd.Series:
    """ETo de referencia (mm/día) por Hargreaves-Samani (FAO-56).

    Ra (radiación extraterrestre) se calcula de latitud + día del año,
    por lo que no requiere sensor de radiación. Reemplaza a Eto_Promedio,
    que en origen es un placeholder (solo 0 o 30) e inservible.
    """
    J = fecha.dt.dayofyear
    phi = np.radians(lat_deg)
    dr = 1 + 0.033 * np.cos(2 * np.pi / 365 * J)
    dec = 0.409 * np.sin(2 * np.pi / 365 * J - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(dec), -1, 1))
    ra_mm = 0.408 * (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(phi) * np.sin(dec)
        + np.cos(phi) * np.cos(dec) * np.sin(ws)
    )
    return (
        0.0023
        * (temp_media + 17.8)
        * np.sqrt((temp_max - temp_min).clip(lower=0))
        * ra_mm
    )

with engine_proy.begin() as conn:
    df_clima = (
        pd.read_sql(query_clima, conn)
        .assign(Fecha=lambda d: pd.to_datetime(d["Fecha"]))
        .assign(
            # Latitud del fundo (con default para los sin coordenada)
            _Lat=lambda d: d["Fundo"].map(LAT_POR_FUNDO).fillna(LAT_DEFAULT),
            # ETo de referencia estimada
            ETo_Hargreaves=lambda d: eto_hargreaves(
                d["Temp_Max"], d["Temp_Min"], d["Temp_Media"], d["Fecha"], d["_Lat"]
            ),
            # Balance hídrico climático (lluvia - ETo referencia). Sin riego todavía.
            Balance_Hidrico_Climatico=lambda d: d["Lluvia_Total"] - d["ETo_Hargreaves"],
        )
        .drop(columns="_Lat")
    )

df_clima.head()

,Fundo,Fecha,Registros_Observados,Registros_Temperatura,Cobertura_Registros,Dia_Valido,Cobertura_Temperatura,Temp_Media,Temp_Max,Temp_Min,Temp_DesvStd,Temp_Media_Dia,Temp_Media_Noche,Horas_Temp_Mayor_28,Horas_Temp_Mayor_30,Horas_Temp_Mayor_32,Horas_Temp_Mayor_35,Horas_Temp_Menor_15,Horas_Temp_Menor_10,HR_Media,HR_Min,HR_Max,HR_DesvStd,HR_Media_Dia,HR_Media_Noche,Horas_HR_Mayor_80,Horas_HR_Mayor_85,Horas_HR_Mayor_90,Horas_HR_Mayor_95,VPD_Medio,VPD_Min,VPD_Max,VPD_DesvStd,VPD_Medio_Dia,VPD_Medio_Noche,Horas_VPD_Alto,Horas_VPD_MuyAlto,VPD_P90,Rad_Solar_Media,Radiacion_Media_Diurna,Rad_Solar_Max,Radiacion_MJ_m2,Horas_Radiacion_Alta,Horas_Luz,Lluvia_Total,Lluvia_Max,Dia_Con_Lluvia,Horas_Con_Lluvia,Vel_Viento_Media,Vel_Viento_Max,Viento_P90,Eva_ET_Total,Punto_Rocio_Medio,Punto_Rocio_Min,Punto_Rocio_Max,Temp_Media_MaxMin,Amplitud_Termica,GDD_Diario,Diferencia_Temp_Dia_Noche,Diferencia_HR_Noche_Dia,Delta_Temp_Punto_Rocio,ETo_Hargreaves,Balance_Hidrico_Climatico
0,AGRICULTOR,2021-01-01,96,96,2.0,1,2.0,22.700000,27.7,19.6,2.434662,24.612000,20.621739,0.0,0.0,0.0,0.0,0.0,0.0,73.760417,58.0,88.0,8.415205,67.500000,80.565217,13.5,3.5,0.0,0.0,0.778819,0.277328,1.561035,0.359891,1.049715,0.484367,8.0,0.5,1.281668,214.468750,411.780000,955.0,18.5301,10.5,25.0,0.0,0.0,0,0.0,4.366667,32.2,12.1,8.10,17.650000,16.3,19.4,23.65,8.1,13.65,3.990261,13.065217,5.050000,4.214525,-4.214525
1,AGRICULTOR,2021-01-02,96,96,2.0,1,2.0,22.715625,29.9,17.7,3.651512,25.314000,19.891304,5.5,0.0,0.0,0.0,0.0,0.0,74.770833,55.0,87.0,9.849904,67.680000,82.478261,22.0,2.5,0.0,0.0,0.790443,0.273079,1.941404,0.496900,1.133659,0.417382,12.0,5.5,1.583931,270.000000,518.360000,932.0,23.3280,13.5,25.0,0.0,0.0,0,0.0,5.257292,32.2,14.5,9.99,17.814583,15.3,19.8,23.80,12.2,13.80,5.422696,14.798261,4.901042,5.175445,-5.175445
2,AGRICULTOR,2021-01-03,96,96,2.0,1,2.0,21.892708,29.1,18.8,2.987456,23.775000,19.668182,3.0,0.0,0.0,0.0,0.0,0.0,79.718750,58.0,89.0,9.008421,74.153846,86.295455,29.5,17.0,0.0,0.0,0.598085,0.241625,1.696946,0.405044,0.833812,0.319499,5.0,2.0,1.247813,221.593750,409.096154,1020.0,19.1457,10.0,26.0,0.0,0.0,0,0.0,3.733333,25.7,11.3,7.96,18.090625,17.0,20.3,23.95,10.3,13.95,4.106818,12.141608,3.802083,4.659848,-4.659848
3,AGRICULTOR,2021-01-04,96,96,2.0,1,2.0,22.356250,29.2,18.8,3.194364,24.835417,19.877083,4.0,0.0,0.0,0.0,0.0,0.0,78.822917,59.0,89.0,9.636548,71.666667,85.979167,29.0,18.0,0.0,0.0,0.648071,0.241625,1.686891,0.442165,0.963963,0.332179,6.5,4.5,1.396012,258.041667,515.875000,991.0,22.2948,12.0,24.0,0.0,0.0,0,0.0,4.271875,29.0,14.5,9.25,18.340625,16.9,20.4,24.00,10.4,14.00,4.958333,14.312500,4.015625,4.738188,-4.738188
4,AGRICULTOR,2021-01-05,96,96,2.0,1,2.0,22.945833,29.8,19.1,3.157411,25.283333,20.608333,4.5,0.0,0.0,0.0,0.0,0.0,76.291667,56.0,88.0,10.061826,69.416667,83.166667,21.5,15.5,0.0,0.0,0.745296,0.270382,1.842866,0.469622,1.068796,0.421796,10.5,4.5,1.468583,210.229167,420.250000,894.0,18.1638,12.5,24.0,0.0,0.0,0,0.0,3.748958,30.6,12.1,8.00,18.365625,17.0,20.2,24.45,10.7,14.45,4.675000,13.750000,4.580208,4.877756,-4.877756


In [15]:
df_clima_semanal = (
    df_clima
    # 1. Agregar información calendario a cada Fecha
    .merge(
        df_calendario_juliano[[ "Fecha", "Año", "Semana", "AñoSemana", "Mes", "NombreMes", "FechaInicioSemana", "FechaFinSemana", "Dias_En_Semana", "Semana_Correlativa"]],
        on="Fecha",
        how="left",
        validate="many_to_one"
    )
    # Solo registros que encontraron correspondencia en calendario
    .loc[lambda d: d["AñoSemana"].notna()]
    # 2. Pasar de Fundo + Fecha a Fundo + Semana
    .groupby(["Fundo", "Año", "Semana", "AñoSemana", "Semana_Correlativa", "Mes", "NombreMes", "FechaInicioSemana", "FechaFinSemana", "Dias_En_Semana"], as_index=False)
    # 3. Agregaciones semanales
    .agg(
        # CALIDAD / COBERTURA
        Dias_Con_Datos=("Fecha", "nunique"),
        Dias_Validos=("Dia_Valido", "sum"),
        Registros_Observados=("Registros_Observados", "sum"),
        Registros_Temperatura=("Registros_Temperatura", "sum"),
        Cobertura_Registros_Prom=("Cobertura_Registros", "mean"),
        Cobertura_Temperatura_Prom=("Cobertura_Temperatura", "mean"),
        # TEMPERATURA
        Temp_Media=("Temp_Media", "mean"),
        Temp_Max=("Temp_Max", "max"),
        Temp_Min=("Temp_Min", "min"),

        Temp_DesvStd_Prom=("Temp_DesvStd", "mean"),

        Temp_Media_Dia=("Temp_Media_Dia", "mean"),
        Temp_Media_Noche=("Temp_Media_Noche", "mean"),

        Horas_Temp_Mayor_28=("Horas_Temp_Mayor_28", "sum"),
        Horas_Temp_Mayor_30=("Horas_Temp_Mayor_30", "sum"),
        Horas_Temp_Mayor_32=("Horas_Temp_Mayor_32", "sum"),
        Horas_Temp_Mayor_35=("Horas_Temp_Mayor_35", "sum"),

        Horas_Temp_Menor_15=("Horas_Temp_Menor_15", "sum"),
        Horas_Temp_Menor_10=("Horas_Temp_Menor_10", "sum"),
        # HUMEDAD RELATIVA
        HR_Media=("HR_Media", "mean"),
        HR_Min=("HR_Min", "min"),
        HR_Max=("HR_Max", "max"),
        HR_DesvStd_Prom=("HR_DesvStd", "mean"),

        HR_Media_Dia=("HR_Media_Dia", "mean"),
        HR_Media_Noche=("HR_Media_Noche", "mean"),

        Horas_HR_Mayor_80=("Horas_HR_Mayor_80", "sum"),
        Horas_HR_Mayor_85=("Horas_HR_Mayor_85", "sum"),
        Horas_HR_Mayor_90=("Horas_HR_Mayor_90", "sum"),
        Horas_HR_Mayor_95=("Horas_HR_Mayor_95", "sum"),
        # VPD
        VPD_Medio=("VPD_Medio", "mean"),
        VPD_Min=("VPD_Min", "min"),
        VPD_Max=("VPD_Max", "max"),
        VPD_DesvStd_Prom=("VPD_DesvStd", "mean"),

        VPD_Medio_Dia=("VPD_Medio_Dia", "mean"),
        VPD_Medio_Noche=("VPD_Medio_Noche", "mean"),

        Horas_VPD_Alto=("Horas_VPD_Alto", "sum"),
        Horas_VPD_MuyAlto=("Horas_VPD_MuyAlto", "sum"),

        VPD_P90_Prom=("VPD_P90", "mean"),
        VPD_P90_Max=("VPD_P90", "max"),
        # RADIACIÓN
        Rad_Solar_Media=("Rad_Solar_Media", "mean"),
        Radiacion_Media_Diurna=("Radiacion_Media_Diurna", "mean"),
        Rad_Solar_Max=("Rad_Solar_Max", "max"),

        # Esta sí es acumulable
        Radiacion_MJ_m2=("Radiacion_MJ_m2", "sum"),

        Horas_Radiacion_Alta=("Horas_Radiacion_Alta", "sum"),
        Horas_Luz=("Horas_Luz", "sum"),
        # LLUVIA
        Lluvia_Total=("Lluvia_Total", "sum"),
        Lluvia_Max=("Lluvia_Max", "max"),
        Dias_Con_Lluvia=("Dia_Con_Lluvia", "sum"),
        Horas_Con_Lluvia=("Horas_Con_Lluvia", "sum"),
        # VIENTO
        Vel_Viento_Media=("Vel_Viento_Media", "mean"),
        Vel_Viento_Max=("Vel_Viento_Max", "max"),

        Viento_P90_Prom=("Viento_P90", "mean"),
        Viento_P90_Max=("Viento_P90", "max"),
        # EVAPOTRANSPIRACIÓN
        Eva_ET_Total=("Eva_ET_Total", "sum"),
        # PUNTO DE ROCÍO
        Punto_Rocio_Medio=("Punto_Rocio_Medio", "mean"),
        Punto_Rocio_Min=("Punto_Rocio_Min", "min"),
        Punto_Rocio_Max=("Punto_Rocio_Max", "max"),
        # VARIABLES DERIVADAS
        Temp_Media_MaxMin=("Temp_Media_MaxMin", "mean"),

        Amplitud_Termica_Media=("Amplitud_Termica", "mean"),
        Amplitud_Termica_Max=("Amplitud_Termica", "max"),

        # GDD es acumulable
        GDD_Semanal=("GDD_Diario", "sum"),

        Diferencia_Temp_Dia_Noche=( "Diferencia_Temp_Dia_Noche", "mean"),

        Diferencia_HR_Noche_Dia=( "Diferencia_HR_Noche_Dia", "mean" ),

        Delta_Temp_Punto_Rocio=( "Delta_Temp_Punto_Rocio", "mean" ),

        # ETo es acumulable
        ETo_Hargreaves=("ETo_Hargreaves", "sum"),

        # Lluvia - ETo también es acumulable
        Balance_Hidrico_Climatico=( "Balance_Hidrico_Climatico", "sum" ),
    )
    # 4. Orden final
    .sort_values( ["Fundo", "Semana_Correlativa"] )
    .reset_index(drop=True)
)

validar_unicidad(df_clima_semanal, ["Fundo", "AñoSemana"], "df_clima_semanal")

df_clima_semanal.head()

,Fundo,Año,Semana,AñoSemana,Semana_Correlativa,Mes,NombreMes,FechaInicioSemana,FechaFinSemana,Dias_En_Semana,Dias_Con_Datos,Dias_Validos,Registros_Observados,Registros_Temperatura,Cobertura_Registros_Prom,Cobertura_Temperatura_Prom,Temp_Media,Temp_Max,Temp_Min,Temp_DesvStd_Prom,Temp_Media_Dia,Temp_Media_Noche,Horas_Temp_Mayor_28,Horas_Temp_Mayor_30,Horas_Temp_Mayor_32,Horas_Temp_Mayor_35,Horas_Temp_Menor_15,Horas_Temp_Menor_10,HR_Media,HR_Min,HR_Max,HR_DesvStd_Prom,HR_Media_Dia,HR_Media_Noche,Horas_HR_Mayor_80,Horas_HR_Mayor_85,Horas_HR_Mayor_90,Horas_HR_Mayor_95,VPD_Medio,VPD_Min,VPD_Max,VPD_DesvStd_Prom,VPD_Medio_Dia,VPD_Medio_Noche,Horas_VPD_Alto,Horas_VPD_MuyAlto,VPD_P90_Prom,VPD_P90_Max,Rad_Solar_Media,Radiacion_Media_Diurna,Rad_Solar_Max,Radiacion_MJ_m2,Horas_Radiacion_Alta,Horas_Luz,Lluvia_Total,Lluvia_Max,Dias_Con_Lluvia,Horas_Con_Lluvia,Vel_Viento_Media,Vel_Viento_Max,Viento_P90_Prom,Viento_P90_Max,Eva_ET_Total,Punto_Rocio_Medio,Punto_Rocio_Min,Punto_Rocio_Max,Temp_Media_MaxMin,Amplitud_Termica_Media,Amplitud_Termica_Max,GDD_Semanal,Diferencia_Temp_Dia_Noche,Diferencia_HR_Noche_Dia,Delta_Temp_Punto_Rocio,ETo_Hargreaves,Balance_Hidrico_Climatico
0,AGRICULTOR,2020,53,2101,54,12,Diciembre,2020-12-28,2021-01-03,7,3,3,288,288,2.0,2.0,22.436111,29.9,17.7,3.024543,24.567000,20.060408,8.5,0.0,0.0,0.0,0.0,0.0,76.083333,55.0,89.0,9.091177,69.777949,83.112978,65.0,23.0,0.0,0.0,0.722449,0.241625,1.941404,0.420612,1.005728,0.407082,25.0,8.0,1.371137,1.583931,235.354167,446.412051,1020.0,61.0038,34.0,76.0,0.0,0.0,0,0.0,4.452431,32.2,12.633333,14.5,26.05,17.851736,15.3,20.3,23.800000,10.200000,12.2,41.40,4.506592,13.335029,4.584375,14.049818,-14.049818
1,AGRICULTOR,2021,1,2102,55,1,Enero,2021-01-04,2021-01-10,7,7,7,672,672,2.0,2.0,22.859673,30.3,18.6,3.109780,25.070474,20.558231,28.5,1.5,0.0,0.0,0.0,0.0,76.745536,52.0,90.0,9.966857,69.924558,83.849563,167.5,85.0,0.0,0.0,0.728618,0.218265,2.132871,0.465431,1.042086,0.402043,64.0,31.0,1.486316,1.756720,223.724702,438.447653,1024.0,135.3087,70.5,171.5,0.0,0.0,0,0.0,3.999256,30.6,11.971429,14.5,59.28,18.380506,16.7,20.4,24.357143,10.685714,11.5,100.50,4.512243,13.925006,4.479167,34.053696,-34.053696
2,AGRICULTOR,2021,2,2103,56,1,Enero,2021-01-11,2021-01-17,7,7,7,672,672,2.0,2.0,23.135417,30.8,18.3,3.123389,25.260845,20.839716,30.5,4.5,0.0,0.0,0.0,0.0,74.355655,51.0,90.0,9.564857,68.104918,81.102002,118.0,26.5,0.0,0.0,0.810665,0.223892,2.229179,0.472924,1.117782,0.478715,76.5,39.0,1.532340,1.818494,227.677083,437.947299,960.0,137.6991,68.0,174.0,0.0,0.0,0,0.0,4.321131,32.2,13.121429,16.1,61.24,18.139286,15.8,20.1,24.421429,10.814286,11.9,100.95,4.421129,12.997085,4.996131,34.476966,-34.476966
3,AGRICULTOR,2021,3,2104,57,1,Enero,2021-01-18,2021-01-24,7,7,7,672,672,2.0,2.0,23.415030,31.6,18.2,2.951919,25.499530,21.258790,31.0,7.5,0.0,0.0,0.0,0.0,75.522321,51.0,87.0,8.998715,69.291733,81.969186,138.0,24.5,0.0,0.0,0.780982,0.274831,2.295562,0.446810,1.083643,0.467974,65.0,34.0,1.475838,1.758937,233.642857,458.535316,1022.0,141.3072,79.0,171.0,0.4,0.2,1,1.0,4.397917,35.4,13.700000,16.1,62.97,18.685565,15.9,21.0,24.878571,10.557143,12.0,104.15,4.240740,12.677453,4.729464,34.403051,-34.003051
4,AGRICULTOR,2021,4,2105,58,1,Enero,2021-01-25,2021-01-31,7,7,7,672,672,2.0,2.0,23.854911,32.4,19.1,3.302594,26.270345,21.356056,56.0,12.5,1.5,0.0,0.0,0.0,72.837798,45.0,88.0,10.449195,65.531190,80.393375,110.5,28.5,0.0,0.0,0.905249,0.294779,2.715750,0.547163,1.282994,0.514245,95.5,62.5,1.780085,2.235969,257.976190,506.762857,1127.0,156.0240,87.5,171.0,0.0,0.0,0,0.0,4.697917,37.0,14.385714,16.9,70.29,18.448810,16.1,21.1,25.428571,11.257143,13.3,108.00,4.914289,14.862184,5.406101,35.930542,-35.930542


In [16]:
# QUERY PRODUCCIÓN REAL
query_produccion_real = f"""
-- df_produccion_real
SELECT
    -- SUBSTRING(t.CampanaAnio, 1, 4) AS Campaña,
    t.Lote,
    CAST(t.fechaProduccion AS DATE) AS Fecha,
    SUM(t.Kilos / 1000.0) AS Toneladas
FROM [Proyecciones].[wr_ProduccionRealPorDiaPorLote_Arandano] t
WHERE 1=1
    -- AND CAST(t.fechaProduccion AS DATE) >= '2020-01-01'
    AND t.fechaProduccion >= '{FECHA_INICIO}'
GROUP BY
    -- SUBSTRING(t.CampanaAnio, 1, 4),
    t.Lote,
    CAST(t.fechaProduccion AS DATE)
"""

# PRODUCCIÓN REAL CORREGIDA
with engine_costos.begin() as conn:
    df_produccion_real = (
        pd.read_sql(query_produccion_real, conn)
        .assign(
            Fecha=lambda d: pd.to_datetime(d['Fecha'])
        )
        # Relacionar producción con todas las campañas posibles del Lote
        .merge(
            df_maestro_lote[[ "Campaña", "Lote", "FechaIniCos", "FechaFinCos", "Ha" ]]
            .assign(
                FechaIniCos=lambda d: pd.to_datetime(d["FechaIniCos"]),
                FechaFinCos=lambda d: pd.to_datetime(d["FechaFinCos"])
            ),
            on="Lote",
            how="left"
        )
        # La fecha de producción debe encontrarse dentro de la ventana de cosecha de la campaña
        .loc[ lambda d: d["Fecha"].between( d["FechaIniCos"], d["FechaFinCos"], inclusive="both" ) ]
        # Resultado final
        [[ "Campaña", "Lote", "Fecha", "Ha", "Toneladas" ]]
        .sort_values( ["Campaña", "Lote", "Fecha"] )
        .reset_index(drop=True)
    )

validar_unicidad(df_produccion_real, ["Campaña", "Lote", "Fecha"], "df_produccion_real")

df_produccion_real.head()

,Campaña,Lote,Fecha,Ha,Toneladas
0,2021.0,L1206,2021-07-30,0.43,0.024188
1,2021.0,L1206,2021-08-05,0.43,0.013440
2,2021.0,L1206,2021-08-10,0.43,0.062760
3,2021.0,L1206,2021-08-16,0.43,0.009189
4,2021.0,L1206,2021-08-21,0.43,0.011502


In [17]:
# 1. Filtra el DataFrame original para conservar las filas con más de una campaña por Fecha y Lote
df_produccion_real_multiple_campaña = df_produccion_real[
    df_produccion_real.groupby(['Fecha', 'Lote'])['Campaña'].transform('nunique') > 1
].sort_values(by=['Fecha', 'Lote']) # Ordenado para ver las campañas duplicadas juntas

df_produccion_real_multiple_campaña
# # 2. Define la ruta usando el patrón pathlib
# ruta_archivo = BASE / "9. ANALISTA_PROYECCION" / "6. ENRIQUE_MOSQUEIRA" / "projects" / "produccion_multiple_campana.xlsx"

# # 3. Exporta a Excel
# df_produccion_real_multiple_campaña.to_excel(ruta_archivo, index=False)

,Campaña,Lote,Fecha,Ha,Toneladas
160111,2022.0,L3752,2023-01-04,0.45,1.825634
258807,2023.0,L3752,2023-01-04,0.45,1.825634
160670,2022.0,L3762,2023-01-04,0.79,2.120326
259203,2023.0,L3762,2023-01-04,0.79,2.120326
160671,2022.0,L3762,2023-01-05,0.79,1.551540
...,...,...,...,...,...
259219,2023.0,L3762,2023-05-12,0.79,0.077576
160127,2022.0,L3752,2023-05-19,0.45,0.058327
258823,2023.0,L3752,2023-05-19,0.45,0.058327
160687,2022.0,L3762,2023-05-19,0.79,0.045367


In [18]:
# df_produccion_real[(df_produccion_real['Lote']=='L4579') & (df_produccion_real['Fecha']=='2026-04-30')]
df_produccion_real[(df_produccion_real['Lote']=='L4372') & (df_produccion_real['Fecha']=='2022-06-23')]

,Campaña,Lote,Fecha,Ha,Toneladas
181591,2022.0,L4372,2022-06-23,0.41,0.006934


In [19]:
df_produccion_semanal = (
    df_produccion_real
    .merge(
        df_calendario_juliano[[ "Fecha", "Año", "Semana", "AñoSemana", "Semana_Correlativa" ]],
        on="Fecha",
        how="left",
        validate="many_to_one"
    )
    .groupby( [ "Campaña", "Lote", "Año", "Semana", "AñoSemana", "Semana_Correlativa" ], as_index=False )
    .agg(
        Toneladas=("Toneladas", "sum"),
        Ha=("Ha", "first")
    )
    .sort_values( ["Campaña", "Lote", "Semana_Correlativa"] )
    .reset_index(drop=True)
    .assign(
        Rendimiento=lambda d: d["Toneladas"].div(d["Ha"]).where(d["Ha"].gt(0)),
    )
)

df_produccion_semanal.head()

,Campaña,Lote,Año,Semana,AñoSemana,Semana_Correlativa,Toneladas,Ha,Rendimiento
0,2021.0,L1206,2021,30,2131,84,0.024188,0.43,0.056251
1,2021.0,L1206,2021,31,2132,85,0.013440,0.43,0.031256
2,2021.0,L1206,2021,32,2133,86,0.062760,0.43,0.145953
3,2021.0,L1206,2021,33,2134,87,0.020691,0.43,0.048119
4,2021.0,L1206,2021,34,2135,88,0.041208,0.43,0.095833


In [20]:
df_peso_real = (
    pd.read_excel(ruta_peso_real, sheet_name = 'Sheet1')
    [['nombre_campaña', 'Parcela', 'Fecha', 'Peso']]
    .rename(columns = {'Parcela': 'SubParcela', 'nombre_campaña':'Campaña'})
    .assign(
        Fecha = lambda d: pd.to_datetime(d['Fecha']),
        Campaña = lambda d: (
            d['Campaña']
            .str.extract(r'^(\d{4})', expand=False)
            .astype('Int64')
        )
    )
    .loc[lambda d: d['Campaña'] >= CAMPANIA]
)

df_peso_real.head()

,Campaña,SubParcela,Fecha,Peso
0,2026,P168A,2026-08-26,3.448276
1,2026,P169A,2026-08-26,3.539823
2,2026,P169A,2026-08-27,3.392857
3,2026,P77,2026-08-25,1.825726
4,2026,P64,2026-08-24,1.396709


In [21]:
df_peso_real_semanal = (
    df_peso_real
    .merge(
        df_calendario_juliano[[ "Fecha", "Año", "Semana", "AñoSemana", "Semana_Correlativa" ]],
        on="Fecha",
        how="left",
        validate="many_to_one"
    )
    .groupby( [ "Campaña", "SubParcela", "Año", "Semana", "AñoSemana", "Semana_Correlativa" ], as_index=False )
    .agg(
        Peso=("Peso", "sum"),
        Peso_Promedio=("Peso", "mean"),
        Peso_Mediana=("Peso", "median"),
        Peso_Min=("Peso", "min"),
        Peso_Max=("Peso", "max"),
        Peso_DesvStd=("Peso", "std"),
        Cantidad_Observaciones=("Peso", "count"),
    )
    .assign(
        Peso_Rango=lambda d: d["Peso_Max"] - d["Peso_Min"],
        Peso_CV=lambda d: d["Peso_DesvStd"].div(d["Peso_Promedio"]).where(d["Peso_Promedio"].ne(0))
    )
    .sort_values( ["Campaña", "SubParcela", "Semana_Correlativa"] )
    .reset_index(drop=True)
)

df_peso_real_semanal.head()

,Campaña,SubParcela,Año,Semana,AñoSemana,Semana_Correlativa,Peso,Peso_Promedio,Peso_Mediana,Peso_Min,Peso_Max,Peso_DesvStd,Cantidad_Observaciones,Peso_Rango,Peso_CV
0,2021,P03,2021,33,2134,87,4.617496,2.308748,2.308748,2.278607,2.338889,0.042626,2,0.060282,0.018463
1,2021,P03,2021,34,2135,88,6.599623,2.199874,2.081146,2.025253,2.493225,0.255582,3,0.467972,0.116180
2,2021,P03,2021,35,2136,89,6.222667,2.074222,2.083732,2.047368,2.091566,0.023584,3,0.044198,0.011370
3,2021,P03,2021,36,2137,90,10.246790,2.049358,2.066194,1.977833,2.085366,0.045012,5,0.107533,0.021964
4,2021,P03,2021,37,2138,91,13.709940,1.958563,1.961905,1.709746,2.129973,0.140551,7,0.420228,0.071762


In [22]:
query_flores_nuevas = f"""
SELECT
    tma.Campaña,
    l.Nombre AS Lote,
    CAST(tma.Fecha AS DATE) AS Fecha,
    SUM(tma.Flores) AS Flores,
    COUNT(DISTINCT(CONCAT(tma.Muestra, '|', tma.Planta))) AS Plantas
FROM TablaMaestraArandano tma
    LEFT JOIN Lote l ON tma.idLote = l.idLote
    LEFT JOIN Variedad v ON v.idVariedad = tma.idVariedad
WHERE
    tma.idEvaluacion = 54
    AND v.idCultivo = 9
    AND tma.Campaña >= {CAMPANIA}
GROUP BY
    tma.Campaña,
    l.Nombre,
    CAST(tma.Fecha AS DATE)
"""

with engine_proy.begin() as conn:
    df_flores_nuevas = (
        pd.read_sql(query_flores_nuevas, conn)
        .assign(
            Fecha = lambda d: pd.to_datetime(d['Fecha']),
        )
    )

df_flores_nuevas.head()

,Campaña,Lote,Fecha,Flores,Plantas
0,2021,L1206,2021-05-15,39,5
1,2021,L1206,2021-05-21,48,5
2,2021,L1206,2021-06-16,337,5
3,2021,L1206,2021-06-22,302,5
4,2021,L1206,2021-08-16,1496,5


In [23]:
df_flores_nuevas_semanal = (
    df_flores_nuevas
    .merge(
        df_calendario_juliano[[ "Fecha", "Año", "Semana", "AñoSemana", "Semana_Correlativa" ]],
        on="Fecha",
        how="left",
        validate="many_to_one"
    )
    .groupby( [ "Campaña", "Lote", "Año", "Semana", "AñoSemana", "Semana_Correlativa" ], as_index=False )
    .agg(
        Flores=("Flores", "sum"),
        Plantas=("Plantas", "sum"),
    )
    .sort_values( ["Campaña", "Lote", "Semana_Correlativa"] )
    .reset_index(drop=True)
    .assign(
        Flores_Acumuladas=lambda d: d.groupby(["Campaña", "Lote"])["Flores"].cumsum(),
        Flores_x_Planta=lambda d: d["Flores"].div(d["Plantas"]).where(d["Plantas"].ne(0))
    )
)

df_flores_nuevas_semanal.head()

,Campaña,Lote,Año,Semana,AñoSemana,Semana_Correlativa,Flores,Plantas,Flores_Acumuladas,Flores_x_Planta
0,2021,L1206,2021,16,2117,70,0,5,0,0.0
1,2021,L1206,2021,17,2118,71,0,5,0,0.0
2,2021,L1206,2021,18,2119,72,34,5,34,6.8
3,2021,L1206,2021,19,2120,73,39,5,73,7.8
4,2021,L1206,2021,20,2121,74,48,5,121,9.6


In [24]:
# CONTROL DE ÚLTIMA SEMANA DISPONIBLE

def _corte(df, fuente, filtro):
    d = df.loc[filtro(df)]
    if d.empty: return {"Fuente": fuente, "Año": np.nan, "Semana": np.nan, "Registros": 0}
    r = d.loc[d["Semana_Correlativa"].idxmax()]
    return {"Fuente": fuente, "Año": int(r["Año"]), "Semana": int(r["Semana"]), "Registros": len(d)}

corte_fuentes = pd.DataFrame([
    _corte(df_produccion_semanal, "Producción", lambda d: d["Campaña"].eq(2026)),
    _corte(df_peso_real_semanal, "Peso", lambda d: d["Campaña"].eq(2026)),
    _corte(df_flores_nuevas_semanal, "Flores", lambda d: d["Campaña"].eq(2026)),
    _corte(df_clima_semanal, "Clima", lambda d: d["Año"].eq(2026))
])

corte_fuentes

,Fuente,Año,Semana,Registros
0,Producción,2026,36,15761
1,Peso,2026,35,637
2,Flores,2026,35,38141
3,Clima,2026,35,541


In [26]:
# DATASET BASE PARA MACHINE LEARNING (Campaña + Lote + AñoSemana)

fecha_corte_produccion = df_produccion_real["Fecha"].max().normalize()

df_consolidado_semanal = (
    df_maestro_semanal
    # PRODUCCIÓN (Campaña + Lote + AñoSemana)
    .merge(
        df_produccion_semanal[[ "Campaña", "Lote", "AñoSemana", "Toneladas"]],
        on=["Campaña", "Lote", "AñoSemana"],
        how="left",
        validate="one_to_one"
    )
    # CLIMA (Fundo + AñoSemana)
    .merge(
        df_clima_semanal.drop(
            columns=[ "Año", "Semana", "Semana_Correlativa", "Mes", "NombreMes", "FechaInicioSemana", "FechaFinSemana", "Dias_En_Semana"],
            errors="ignore"
        ),
        on=["Fundo", "AñoSemana"],
        how="left",
        validate="many_to_one"
    )
    # PESO (Campaña + SubParcela + AñoSemana)
    .merge(
        df_peso_real_semanal[[ "Campaña", "SubParcela", "AñoSemana", "Peso_Promedio", "Peso_Mediana", "Peso_Min", "Peso_Max", "Peso_DesvStd", "Cantidad_Observaciones", "Peso_Rango", "Peso_CV"]],
        on=["Campaña", "SubParcela", "AñoSemana"],
        how="left",
        validate="many_to_one"
    )
    # FLORES (Campaña + Lote + AñoSemana)
    .merge(
        df_flores_nuevas_semanal[[ "Campaña", "Lote", "AñoSemana", "Flores", "Plantas", "Flores_Acumuladas", "Flores_x_Planta"]],
        on=["Campaña", "Lote", "AñoSemana"],
        how="left",
        validate="one_to_one"
    )
    # ORDEN TEMPORAL
    .sort_values(["Campaña", "Lote", "Semana_Correlativa"])
    .reset_index(drop=True)
    # VARIABLES TEMPORALES
    .assign(
        Semana_Campaña=lambda d: d["Semana_Correlativa"] - d.groupby( ["Campaña", "Lote"] )["Semana_Correlativa"].transform("min") + 1,
        Dias_Desde_Inicio_Cosecha=lambda d: ( d["FechaInicioSemana"] - d["FechaIniCos"] ).dt.days,
        Semana_Sin=lambda d: np.sin(2 * np.pi * d["Semana"] / 52),
        Semana_Cos=lambda d: np.cos(2 * np.pi * d["Semana"] / 52),
        # Para campañas históricas:
        # ausencia de producción = producción cero.
        #
        # Para semana actual incompleta o futuras:
        # dejamos NaN para no entrenar con información parcial.
        Toneladas=lambda d:
            d["Toneladas"]
            .fillna(0)
            .where(
                d["FechaFinSemana"] <= fecha_corte_produccion
            )
    )
    # TARGET
    .assign(
        Rendimiento=lambda d: d["Toneladas"].div(d["Ha"]).where(d["Ha"].gt(0))
    )
)

validar_unicidad( df_consolidado_semanal, ["Campaña", "Lote", "AñoSemana"], "df_consolidado_semanal")

validar_cruce( df_maestro_semanal, df_consolidado_semanal, "Construcción df_consolidado_semanal" )

print(
    f"✅ df_consolidado_semanal: {len(df_consolidado_semanal):,} filas | "
    f"{df_consolidado_semanal['Campaña'].nunique()} campañas | "
    f"{df_consolidado_semanal['Lote'].nunique():,} lotes"
)

df_consolidado_semanal.head()

✅ [Construcción df_consolidado_semanal] Validación exitosa: El número de filas se mantuvo intacto en 792068.
✅ df_consolidado_semanal: 792,068 filas | 6 campañas | 2,593 lotes


,Campaña,Variedad,Fundo,Parcela,SubParcela,Lote,PodaEstacion,TipoCultivo,FechaIniCos,FechaFinCos,Nro_Plantas,Ha,FechaInicioCampaña,Densidad_Plantas,Duracion_Cosecha,Periodo_Poda,Anio_Poda,Semana_Poda,Numero_Campaña_Lote,Años_Desde_1ra_Campaña,Semana_Correlativa_Poda,Semana_Correlativa,Año,Semana,AñoSemana,FechaInicioSemana,FechaFinSemana,Semanas_Desde_Poda,Toneladas,Dias_Con_Datos,Dias_Validos,Registros_Observados,Registros_Temperatura,Cobertura_Registros_Prom,Cobertura_Temperatura_Prom,Temp_Media,Temp_Max,Temp_Min,Temp_DesvStd_Prom,Temp_Media_Dia,Temp_Media_Noche,Horas_Temp_Mayor_28,Horas_Temp_Mayor_30,Horas_Temp_Mayor_32,Horas_Temp_Mayor_35,Horas_Temp_Menor_15,Horas_Temp_Menor_10,HR_Media,HR_Min,HR_Max,HR_DesvStd_Prom,HR_Media_Dia,HR_Media_Noche,Horas_HR_Mayor_80,Horas_HR_Mayor_85,Horas_HR_Mayor_90,Horas_HR_Mayor_95,VPD_Medio,VPD_Min,VPD_Max,VPD_DesvStd_Prom,VPD_Medio_Dia,VPD_Medio_Noche,Horas_VPD_Alto,Horas_VPD_MuyAlto,VPD_P90_Prom,VPD_P90_Max,Rad_Solar_Media,Radiacion_Media_Diurna,Rad_Solar_Max,Radiacion_MJ_m2,Horas_Radiacion_Alta,Horas_Luz,Lluvia_Total,Lluvia_Max,Dias_Con_Lluvia,Horas_Con_Lluvia,Vel_Viento_Media,Vel_Viento_Max,Viento_P90_Prom,Viento_P90_Max,Eva_ET_Total,Punto_Rocio_Medio,Punto_Rocio_Min,Punto_Rocio_Max,Temp_Media_MaxMin,Amplitud_Termica_Media,Amplitud_Termica_Max,GDD_Semanal,Diferencia_Temp_Dia_Noche,Diferencia_HR_Noche_Dia,Delta_Temp_Punto_Rocio,ETo_Hargreaves,Balance_Hidrico_Climatico,Peso_Promedio,Peso_Mediana,Peso_Min,Peso_Max,Peso_DesvStd,Cantidad_Observaciones,Peso_Rango,Peso_CV,Flores,Plantas,Flores_Acumuladas,Flores_x_Planta,Semana_Campaña,Dias_Desde_Inicio_Cosecha,Semana_Sin,Semana_Cos,Rendimiento
0,2021,Biloxi,GLORIA,P18B,P18B,L1206,Verano,Convencional,2021-07-30,2022-01-15,2610.0,0.43,2021-02-04,6069.767442,170.0,202105.0,2021.0,5.0,1,1,59.0,59,2021,5,2106,2021-02-01,2021-02-07,0.0,0.0,7.0,7.0,334.0,334.0,0.994048,0.994048,22.754801,32.2,16.5,3.668795,25.478117,19.967857,20.5,4.0,0.0,0.0,0.0,0.0,78.205487,50.0,96.0,12.407164,69.383106,87.241460,89.0,63.5,36.0,2.0,0.713241,0.077789,2.426055,0.552137,1.103782,0.313769,39.5,22.5,1.595115,2.165075,225.460274,445.688064,965.0,136.358374,35.5,84.5,0.0,0.0,0.0,0.0,3.431263,25.7,8.751429,11.30,28.33,18.497399,15.5,21.1,24.114286,11.685714,12.9,98.80,5.510260,17.858354,4.257402,35.650695,-35.650695,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,-179,0.568065,0.822984,0.0
1,2021,Biloxi,GLORIA,P18B,P18B,L1206,Verano,Convencional,2021-07-30,2022-01-15,2610.0,0.43,2021-02-04,6069.767442,170.0,202105.0,2021.0,5.0,1,1,59.0,60,2021,6,2107,2021-02-08,2021-02-14,1.0,0.0,7.0,7.0,336.0,335.0,1.000000,0.997024,24.299018,31.8,17.5,3.707557,27.153572,21.342702,41.0,12.5,0.0,0.0,0.0,0.0,75.356510,52.0,96.0,12.806812,65.637381,85.428830,70.5,46.0,27.5,2.0,0.879309,0.084024,2.288551,0.624050,1.345418,0.397975,54.0,39.0,1.844159,2.119583,243.156345,478.202681,980.0,147.060957,42.0,85.0,0.0,0.0,0.0,0.0,3.867958,29.0,10.284286,11.78,32.51,19.354509,16.5,21.3,25.235714,11.357143,13.7,106.65,5.810871,19.791449,4.944510,36.363839,-36.363839,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,-172,0.663123,0.748511,0.0
2,2021,Biloxi,GLORIA,P18B,P18B,L1206,Verano,Convencional,2021-07-30,2022-01-15,2610.0,0.43,2021-02-04,6069.767442,170.0,202105.0,2021.0,5.0,1,1,59.0,61,2021,7,2108,2021-02-15,2021-02-21,2.0,0.0,7.0,7.0,336.0,336.0,1.000000,1.000000,23.048810,31.8,18.3,3.729615,25.869310,20.192883,27.0,7.5,0.0,0.0,0.0,0.0,77.622024,49.0,97.0,13.225804,68.330714,87.034161,86.0,68.0,41.0,4.0,0.757620,0.065898,2.389258,0.602156,1.185836,0.323866,45.5,30.0,1.708945,2.190172,263.160714,522.977381,971.0,159.159600,44.5,84.5,0.0,0.0,0.0,0.0,3.963393,29.0,9.744286,11.30,33.11,18.592560,17.0,21.5,24.450000,11.242857,12.2,101.15,5.676427,18.703447,4.456250,35.116271,-35.116271,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,-165,0.748511,0.663123,0.0
3,2021,Biloxi,GLORIA,P18B,P18B,L1206,Verano,Convencional,2021-07-30,2022-01-15,2610.0,0.43,2021-02-04,6069.767442,170.0,202105.0,2021.0,5.

In [27]:
df_modelo = (
    df_consolidado_semanal
    .sort_values(["Campaña", "Lote", "Semana_Correlativa"])
    .assign(
        # FLORES
        Flores_Lag_6=lambda d: d.groupby(["Campaña", "Lote"])["Flores_x_Planta"].shift(6),
        Flores_Lag_7=lambda d: d.groupby(["Campaña", "Lote"])["Flores_x_Planta"].shift(7),
        Flores_Lag_8=lambda d: d.groupby(["Campaña", "Lote"])["Flores_x_Planta"].shift(8),
        Flores_Lag_9=lambda d: d.groupby(["Campaña", "Lote"])["Flores_x_Planta"].shift(9),
        Flores_Lag_10=lambda d: d.groupby(["Campaña", "Lote"])["Flores_x_Planta"].shift(10),
        # PESO
        Peso_Lag_1=lambda d: d.groupby(["Campaña", "Lote"])["Peso_Promedio"].shift(1),
        Peso_Lag_2=lambda d: d.groupby(["Campaña", "Lote"])["Peso_Promedio"].shift(2),
        Peso_Lag_3=lambda d: d.groupby(["Campaña", "Lote"])["Peso_Promedio"].shift(3),
        # CLIMA: información disponible hasta semana anterior
        GDD_Lag_1=lambda d: d.groupby(["Campaña", "Lote"])["GDD_Semanal"].shift(1),
        VPD_Lag_1=lambda d: d.groupby(["Campaña", "Lote"])["VPD_Medio"].shift(1),
        Temp_Lag_1=lambda d: d.groupby(["Campaña", "Lote"])["Temp_Media"].shift(1),
        Radiacion_Lag_1=lambda d: d.groupby(["Campaña", "Lote"])["Radiacion_MJ_m2"].shift(1)
    )
)

In [28]:
df_modelo = (
    df_modelo
    .assign(

        Flores_Lag_6_10_Prom=lambda d: d[[f"Flores_Lag_{i}" for i in range(6, 11)]].mean(axis=1),
        Peso_Lag_1_3_Prom=lambda d: d[[f"Peso_Lag_{i}" for i in range(1, 4)]].mean(axis=1),
        GDD_4Sem=lambda d: d.groupby(["Campaña", "Lote"])["GDD_Semanal"].transform(lambda x: x.shift(1).rolling(4, min_periods=1).sum()),
        VPD_4Sem=lambda d: d.groupby(["Campaña", "Lote"])["VPD_Medio"].transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean()),
        Temp_4Sem=lambda d: d.groupby(["Campaña", "Lote"])["Temp_Media"].transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean()),
        Radiacion_4Sem=lambda d: d.groupby(["Campaña", "Lote"])["Radiacion_MJ_m2"].transform(lambda x: x.shift(1).rolling(4, min_periods=1).sum()),
        GDD_Acumulado=lambda d: d.groupby(["Campaña", "Lote"])["GDD_Semanal"].transform(lambda x: x.shift(1).fillna(0).cumsum())
    )
)

¿qué puede predecir hoy tu notebook?

| Periodo 2026      | Qué significa actualmente                                                                            |
| ----------------- | ---------------------------------------------------------------------------------------------------- |
| Hasta S35         | **Predicción histórica 2026**, puedes comparar Real vs Predicho                                      |
| S36               | **Forecast real 1 semana adelante**, con prácticamente todos los predictors disponibles              |
| S37-S38           | Forecast posible, pero ya empiezan a faltar variables dinámicas                                      |
| S39+              | Cada vez depende más de imputaciones                                                                 |
| Hasta FechaFinCos | Python producirá números, **pero todavía no lo considero un verdadero forecast de campaña completa** |

Hay otra característica interesante: Flores sí te da más horizonte

Como utilizamos:

Flores_Lag_6
Flores_Lag_7
Flores_Lag_8
Flores_Lag_9
Flores_Lag_10

si conocemos flores hasta S35, todavía podemos encontrar señales para varias semanas futuras.

Por ejemplo:

Rendimiento S36 ← Flores S26-S30
Rendimiento S37 ← Flores S27-S31
Rendimiento S38 ← Flores S28-S32
Rendimiento S39 ← Flores S29-S33
Rendimiento S40 ← Flores S30-S34
Rendimiento S41 ← Flores S31-S35

Hasta S41, los cinco lags de flores todavía pueden construirse completamente con información conocida a S35.

Por eso agronómicamente tu fuente de flores tiene mucho valor para anticipación.

El problema de horizonte corto lo están ocasionando principalmente:

Peso_Lag_1
Clima_Lag_1

8. Para tu verdadero objetivo, cambiaría ligeramente el planteamiento

Tu objetivo no es solamente:

“¿cuánto rendirá la próxima semana?”

Es:

“Al cierre de la semana 35, quiero ver el rendimiento proyectado de todas las semanas restantes de campaña 2026.”

Eso requiere que el modelo conozca el horizonte de pronóstico:

$$ Horizonte = SemanaObjetivo - SemanaCorte $$

Por ejemplo:

Semana corte = 35

Objetivo S36 → Horizonte 1
Objetivo S37 → Horizonte 2
Objetivo S38 → Horizonte 3
...
Objetivo S50 → Horizonte 15

La solución que recomiendo para tu caso es evolucionar hacia un modelo multi-horizonte.

No intentaría inventar:

Peso S36
Peso S37
Clima S36
Clima S37

para después alimentar recursivamente el rendimiento.

El ajuste conceptual es:

HOY
modelo semanal t+1
        ↓
puede producir toda 2026
pero con imputaciones futuras

OBJETIVO
modelo multi-horizonte
        ↓
Corte S35
        ↓
predice S36, S37, S38 ... FechaFinCos
        ↓
actualizas nuevamente al cerrar S36
        ↓
vuelve a predecir S37 ... FechaFinCos

Ese es el enfoque que recomiendo para el objetivo real de proyectar semanalmente toda la campaña de arándano y actualizar el forecast cada semana.

In [29]:
# Preparar el dataset que realmente alimentará al ML
# Importante: primero hicimos los lags utilizando todas las semanas, incluso las anteriores a cosecha. Ahora podemos limitar el target a semanas correspondientes a la ventana de cosecha.
# Así las semanas previas aportaron información para Flores_Lag_6...10, pero no llenarán el entrenamiento de rendimientos cero antes de iniciar cosecha.

TARGET = "Rendimiento"

df_ml = (
    df_modelo
    .loc[
        df_modelo["FechaInicioSemana"].le(df_modelo["FechaFinCos"]) &
        df_modelo["FechaFinSemana"].ge(df_modelo["FechaIniCos"])
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Filas para modelado: {len(df_ml):,}")
print(f"Campañas: {df_ml['Campaña'].nunique()}")
print(f"Lotes: {df_ml['Lote'].nunique():,}")

Filas para modelado: 493,361
Campañas: 6
Lotes: 2,593


In [30]:
# Definir exactamente qué columnas utilizará el modelo
# Toneladas está absolutamente prohibido como X porque: $$ Rendimiento=\frac{Toneladas}{Ha} $$
# Meter Toneladas sería leakage directo.

num_candidatas = [
    "Ha", "Nro_Plantas", "Densidad_Plantas", "Semana", "Semana_Campaña",
    "Semanas_Desde_Poda", "Dias_Desde_Poda", "Semana_Poda", "Semana_Sin", "Semana_Cos",
    "Flores_Lag_6", "Flores_Lag_7", "Flores_Lag_8", "Flores_Lag_9", "Flores_Lag_10", "Flores_Lag_6_10_Prom",
    "Peso_Lag_1", "Peso_Lag_2", "Peso_Lag_3", "Peso_Lag_1_3_Prom",
    "GDD_Lag_1", "VPD_Lag_1", "Temp_Lag_1", "Radiacion_Lag_1",
    "GDD_4Sem", "VPD_4Sem", "Temp_4Sem", "Radiacion_4Sem", "GDD_Acumulado"
]

cat_candidatas = [
    "Variedad", "Fundo", "Parcela", "SubParcela", "PodaEstacion", "TipoCultivo", "Periodo_Poda"
]

num_cols = [c for c in num_candidatas if c in df_ml.columns and df_ml[c].notna().any()]
cat_cols = [c for c in cat_candidatas if c in df_ml.columns and df_ml[c].notna().any()]
features_modelo = num_cols + cat_cols

df_ml[num_cols] = df_ml[num_cols].apply(pd.to_numeric, errors="coerce")
df_ml[cat_cols] = df_ml[cat_cols].astype("object").where(df_ml[cat_cols].notna(), np.nan)

print(f"Variables numéricas: {len(num_cols)}")
print(f"Variables categóricas: {len(cat_cols)}")
print(f"Total variables modelo: {len(features_modelo)}")

Variables numéricas: 28
Variables categóricas: 7
Total variables modelo: 35


In [33]:
# Separar Train / Validation / Test por campaña

CAMPANIA_PRED = 2026

campanas_hist = sorted(df_ml.loc[df_ml["Campaña"].lt(CAMPANIA_PRED), "Campaña"].dropna().astype(int).unique())
campanas_train = campanas_hist[:-2]
campana_val = campanas_hist[-2]
campana_test = campanas_hist[-1]
campanas_prediccion = [CAMPANIA_PRED]

print("TRAIN:", campanas_train)
print("VALIDATION:", campana_val)
print("TEST:", campana_test)
print("PREDICCIÓN:", campanas_prediccion)

TRAIN: [np.int64(2021), np.int64(2022), np.int64(2023)]
VALIDATION: 2024
TEST: 2025
PREDICCIÓN: [2026]


In [34]:
# Construir X e y

def obtener_xy(campanas):
    d = df_ml.loc[df_ml["Campaña"].isin(np.atleast_1d(campanas)) & df_ml[TARGET].notna()].copy()
    return d, d[features_modelo], d[TARGET]

df_train, X_train, y_train = obtener_xy(campanas_train)
df_val, X_val, y_val = obtener_xy(campana_val)
df_test, X_test, y_test = obtener_xy(campana_test)

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,} | Test: {len(X_test):,}")

Train: 196,184 | Validation: 60,331 | Test: 80,463


In [35]:
# Preparar variables numéricas/categóricas y modelos candidatos

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor

preprocesador = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ]), cat_cols)
])

modelos = {
    "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, l2_regularization=1, random_state=42)
}

try:
    from xgboost import XGBRegressor
    modelos["XGBoost"] = XGBRegressor(n_estimators=500, max_depth=7, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror", n_jobs=-1, random_state=42)
except ImportError:
    pass

print("Modelos:", list(modelos))

Modelos: ['RandomForest', 'ExtraTrees', 'HistGradientBoosting', 'XGBoost']


In [36]:
# Comparar modelos en Validation
# Utilizaremos como principal criterio el WAPE, acompañado de MAE, RMSE y R².

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metricas(y, pred):
    return {
        "WAPE_%": 100 * np.abs(y - pred).sum() / np.abs(y).sum(),
        "MAE": mean_absolute_error(y, pred),
        "RMSE": mean_squared_error(y, pred) ** 0.5,
        "R2": r2_score(y, pred)
    }

resultados = []

for nombre, estimador in modelos.items():
    modelo = Pipeline([("prep", clone(preprocesador)), ("modelo", clone(estimador))])
    modelo.fit(X_train, y_train)
    pred = np.clip(modelo.predict(X_val), 0, None)
    resultados.append({"Modelo": nombre, **metricas(y_val, pred)})

ranking_modelos = (
    pd.DataFrame(resultados)
    .sort_values(["WAPE_%", "RMSE"])
    .reset_index(drop=True)
)

ranking_modelos

,Modelo,WAPE_%,MAE,RMSE,R2
0,HistGradientBoosting,71.050385,0.521392,0.884960,0.296534
1,XGBoost,72.876662,0.534794,0.885152,0.296229
2,RandomForest,75.741547,0.555817,0.923061,0.234657
3,ExtraTrees,77.841605,0.571228,0.903964,0.265996


In [37]:
modelo_ganador_nombre = ranking_modelos.loc[0, "Modelo"]

print("🏆 Modelo seleccionado:", modelo_ganador_nombre)
print(f"WAPE Validation: {ranking_modelos.loc[0, 'WAPE_%']:.2f}%")
print(f"RMSE Validation: {ranking_modelos.loc[0, 'RMSE']:.4f}")

🏆 Modelo seleccionado: HistGradientBoosting
WAPE Validation: 71.05%
RMSE Validation: 0.8850


In [38]:
# Evaluar el ganador contra TEST
# evaluación mucho más realista y métrica mucho más importante que el rendimiento sobre Train.

df_train_val, X_train_val, y_train_val = obtener_xy(campanas_train + [campana_val])

modelo_test = Pipeline([
    ("prep", clone(preprocesador)),
    ("modelo", clone(modelos[modelo_ganador_nombre]))
])

modelo_test.fit(X_train_val, y_train_val)
pred_test = np.clip(modelo_test.predict(X_test), 0, None)

pd.DataFrame([{
    "Modelo": modelo_ganador_nombre,
    "Campaña_Test": campana_test,
    **metricas(y_test, pred_test)
}])

,Modelo,Campaña_Test,WAPE_%,MAE,RMSE,R2
0,HistGradientBoosting,2025,55.373688,0.332033,0.577277,0.466379


In [ ]:
# Ver Real vs Predicho de la campaña Test

df_resultado_test = (
    df_test[["Campaña", "SubParcela", "Lote", "Ha", "Año", "Semana", "Toneladas", TARGET]]
    .assign(
        Rendimiento_Predicho=pred_test,
        Toneladas_Predichas=lambda d: d["Rendimiento_Predicho"] * d["Ha"],
    )
    .rename(columns={
        "Toneladas": "Toneladas_Reales",
        TARGET: "Rendimiento_Real",
    })
    .sort_values(["Campaña", "Lote", "Año", "Semana"])
    .reset_index(drop=True)
)

df_resultado_test.head(20)

In [41]:
# Entrenar finalmente con todas las campañas históricas cerradas
# Una vez que hemos seleccionado el modelo y comprobado su rendimiento en Test, podemos aprovechar toda la historia.

df_historico, X_historico, y_historico = obtener_xy(campanas_hist)

modelo_final = Pipeline([
    ("prep", clone(preprocesador)),
    ("modelo", clone(modelos[modelo_ganador_nombre]))
])

modelo_final.fit(X_historico, y_historico)

print(f"✅ Modelo final: {modelo_ganador_nombre}")
print(f"✅ Filas entrenamiento: {len(X_historico):,}")
print(f"✅ Campañas entrenamiento: {campanas_hist}")

✅ Modelo final: HistGradientBoosting
✅ Filas entrenamiento: 336,978
✅ Campañas entrenamiento: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [42]:
# Predecir la campaña actual
# Aquí aparecen los dos dataframes que aclaraban tu segunda pregunta.
# Primero generamos el dataframe completo, con todas las variables:

df_predicciones_detalle = (
    df_ml.loc[df_ml["Campaña"].isin(campanas_prediccion)]
    .copy()
    .assign(
        Rendimiento_Predicho=lambda d: np.clip(modelo_final.predict(d[features_modelo]), 0, None)
    )
    .sort_values(["Campaña", "Lote", "Año", "Semana"])
    .reset_index(drop=True)
)

df_predicciones_detalle.head()

,Campaña,Variedad,Fundo,Parcela,SubParcela,Lote,PodaEstacion,TipoCultivo,FechaIniCos,FechaFinCos,Nro_Plantas,Ha,FechaInicioCampaña,Densidad_Plantas,Duracion_Cosecha,Periodo_Poda,Anio_Poda,Semana_Poda,Numero_Campaña_Lote,Años_Desde_1ra_Campaña,Semana_Correlativa_Poda,Semana_Correlativa,Año,Semana,AñoSemana,FechaInicioSemana,FechaFinSemana,Semanas_Desde_Poda,Toneladas,Dias_Con_Datos,Dias_Validos,Registros_Observados,Registros_Temperatura,Cobertura_Registros_Prom,Cobertura_Temperatura_Prom,Temp_Media,Temp_Max,Temp_Min,Temp_DesvStd_Prom,Temp_Media_Dia,Temp_Media_Noche,Horas_Temp_Mayor_28,Horas_Temp_Mayor_30,Horas_Temp_Mayor_32,Horas_Temp_Mayor_35,Horas_Temp_Menor_15,Horas_Temp_Menor_10,HR_Media,HR_Min,HR_Max,HR_DesvStd_Prom,HR_Media_Dia,HR_Media_Noche,Horas_HR_Mayor_80,Horas_HR_Mayor_85,Horas_HR_Mayor_90,Horas_HR_Mayor_95,VPD_Medio,VPD_Min,VPD_Max,VPD_DesvStd_Prom,VPD_Medio_Dia,VPD_Medio_Noche,Horas_VPD_Alto,Horas_VPD_MuyAlto,VPD_P90_Prom,VPD_P90_Max,Rad_Solar_Media,Radiacion_Media_Diurna,Rad_Solar_Max,Radiacion_MJ_m2,Horas_Radiacion_Alta,Horas_Luz,Lluvia_Total,Lluvia_Max,Dias_Con_Lluvia,Horas_Con_Lluvia,Vel_Viento_Media,Vel_Viento_Max,Viento_P90_Prom,Viento_P90_Max,Eva_ET_Total,Punto_Rocio_Medio,Punto_Rocio_Min,Punto_Rocio_Max,Temp_Media_MaxMin,Amplitud_Termica_Media,Amplitud_Termica_Max,GDD_Semanal,Diferencia_Temp_Dia_Noche,Diferencia_HR_Noche_Dia,Delta_Temp_Punto_Rocio,ETo_Hargreaves,Balance_Hidrico_Climatico,Peso_Promedio,Peso_Mediana,Peso_Min,Peso_Max,Peso_DesvStd,Cantidad_Observaciones,Peso_Rango,Peso_CV,Flores,Plantas,Flores_Acumuladas,Flores_x_Planta,Semana_Campaña,Dias_Desde_Inicio_Cosecha,Semana_Sin,Semana_Cos,Rendimiento,Flores_Lag_6,Flores_Lag_7,Flores_Lag_8,Flores_Lag_9,Flores_Lag_10,Peso_Lag_1,Peso_Lag_2,Peso_Lag_3,GDD_Lag_1,VPD_Lag_1,Temp_Lag_1,Radiacion_Lag_1,Flores_Lag_6_10_Prom,Peso_Lag_1_3_Prom,GDD_4Sem,VPD_4Sem,Temp_4Sem,Radiacion_4Sem,GDD_Acumulado,Rendimiento_Predicho
0,2026,Maia Blue,GLORIA,P18B,P18B,L1206,Verano,Convencional,2027-02-01,2028-02-01,2614.0,0.43,2027-02-01,6079.069767,366.0,202624.0,2026.0,24.0,6,6,342.0,377,2027,5,2706,2027-02-01,2027-02-07,35.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0.568065,0.822984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.334672
1,2026,Maia Blue,GLORIA,P18B,P18B,L1206,Verano,Convencional,2027-02-01,2028-02-01,2614.0,0.43,2027-02-01,6079.069767,366.0,202624.0,2026.0,24.0,6,6,342.0,378,2027,6,2707,2027-02-08,2027-02-14,36.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,7,0.663123,0.748511,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.209020
2,2026,Maia Blue,GLORIA,P18B,P18B,L1206,Verano,Convencional,2027-02-01,2028-02-01,2614.0,0.43,2027-02-01,6079.069767,366.0,202624.0,2026.0,24.0,6,6,342.0,379,2027,7,2708,2027-02-15,2027-02-21,37.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,14,0.748511,0.663123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.182738
3,2026,Maia Blue,GLORIA,P18B,P18B,L1206,Verano,Convencional,2027-02-01,2028-02-01,2614.0,0.43,2027-02-01,6079.069767,366.0,202624.0,2026.0,24.0,6,6,342.0,380,2027,8,2709,2027-02-22,2027-02-28,38.0,NaN,NaN,NaN,NaN,NaN,NaN

In [46]:
# Resultado final que usarías operativamente
# A partir del dataframe anterior simplemente seleccionamos las seis columnas que necesitas.

df_resultado_final = (
    df_predicciones_detalle[["Campaña", "SubParcela", "Lote", "Ha", "Año", "Semana", "Toneladas", TARGET, "Rendimiento_Predicho"]]
    .rename(columns={
        "Toneladas": "Toneladas Reales",
        TARGET: "Rendimiento Real",
        "Rendimiento_Predicho": "Rendimiento Predicho"
    })
    .assign(
        **{
            "Toneladas Predichas": lambda d:
                d["Rendimiento Predicho"] * d["Ha"]
        }
    )
    [[ "Campaña", "SubParcela", "Lote", "Ha", "Año", "Semana", "Toneladas Reales", "Toneladas Predichas", "Rendimiento Real", "Rendimiento Predicho" ]]
    .reset_index(drop=True)
)

# Y ésta sería tu salida operativa definitiva: Campaña + Lote + Año + Semana + Rendimiento Real + Rendimiento Predicho
df_resultado_final.to_excel(ruta_resultado_final, index=False)

df_resultado_final.head(20)

,Campaña,SubParcela,Lote,Ha,Año,Semana,Toneladas Reales,Toneladas Predichas,Rendimiento Real,Rendimiento Predicho
0,2026,P18B,L1206,0.43,2027,5,NaN,0.143909,NaN,0.334672
1,2026,P18B,L1206,0.43,2027,6,NaN,0.089878,NaN,0.209020
2,2026,P18B,L1206,0.43,2027,7,NaN,0.078577,NaN,0.182738
3,2026,P18B,L1206,0.43,2027,8,NaN,0.078577,NaN,0.182738
4,2026,P18B,L1206,0.43,2027,9,NaN,0.076026,NaN,0.176806
5,2026,P18B,L1206,0.43,2027,10,NaN,0.071526,NaN,0.166340
6,2026,P18B,L1206,0.43,2027,11,NaN,0.070783,NaN,0.164612
7,2026,P18B,L1206,0.43,2027,12,NaN,0.055760,NaN,0.129674
8,2026,P18B,L1206,0.43,2027,13,NaN,0.056428,NaN,0.131229
9,2026,P18B,L1206,0.43,2027,14,NaN,0.055736,NaN,0.129619
